In [1]:
import os
os.listdir('.')

['.kernel_llm_logs_1.txt',
 '.config',
 'memory',
 ' v6 — Binding Context Document.pdf',
 '.prompts']

In [2]:
import mpmath as mp
mp.mp.dps = 30
# Davenport-Heilbronn function
# L_DH(s) = (1/2) * (L(s, chi) + e^{i*theta} * L(s, conjugate_chi))?
# Actually canonical: L_DH(s) = (1 - i tan(theta)) L(s, chi_5) + (1 + i tan(theta)) L(s, chibar)
# Using chi_5 mod 5 nontrivial chi (or chi mod 5), with kappa as given.
# Standard formula: f(s) = sum_{n>=1} c_n / n^s where c_n is periodic mod 5 with values
# 1, kappa, -kappa, -1, 0 (period 5)
# Let's go with that — it's the canonical DH series.
# Then kappa is determined by requiring the functional equation.
# kappa = (sqrt(10 - 2 sqrt(5)) - 2)/(sqrt(5)-1)? Actually canonical value:
# kappa = (sqrt(10 - 2 sqrt(5)) - 2)/(sqrt(5)-1) approximately.
# The validated value mentioned: kappa ~ 0.28408
# Let's verify.

# Standard formula for DH: c_n periodic with c_1=1, c_2=κ, c_3=-κ, c_4=-1, c_5=0
def L_DH(s, kappa, N=100000):
 # Kahan summation
 total = mp.mpc(0)
 c = mp.mpc(0)
 coeffs = [mp.mpf(1), mp.mpf(kappa), -mp.mpf(kappa), mp.mpf(-1), mp.mpf(0)]
 for n in range(1, N+1):
 cn = coeffs[(n-1) % 5]
 if cn == 0:
 continue
 y = cn / mp.power(n, s) - c
 t = total + y
 c = (t - total) - y
 total = t
 return total

# Try kappa = 0.28408
# Canonical zeros (claimed): t ≈ 2.80, 6.32 (off-line zeros at sigma > 1/2)
# Actually DH has zeros off the critical line. Let me first verify the value.
# Standard exact: kappa = sqrt((10 - 2 sqrt(5)))/2 - ... 
# tan(theta) where theta = (something)
# From Titchmarsh: kappa such that 1 + kappa*xi + kappa*xi^2 ... 
# The functional equation requires:
# κ = (sqrt(10 - 2 sqrt(5)) - 2)/(sqrt(5) - 1) — let's compute
val = (mp.sqrt(10 - 2*mp.sqrt(5)) - 2)/(mp.sqrt(5)-1)
print("formula 1:", val)
val2 = mp.tan(mp.pi/5 - mp.atan(mp.sqrt(5-2*mp.sqrt(5))))
print("test:", val2)

formula 1: 0.284079043840412296028291832393
test: -9.86076131526264756764660706603e-32


In [3]:
kappa = (mp.sqrt(10 - 2*mp.sqrt(5)) - 2)/(mp.sqrt(5)-1)
print("kappa =", kappa)

# The DH canonical off-line zeros (real-part > 1/2). According to Spira/Balanzario:
# t ≈ 85.69934, 100.665... etc.
# Actually the known DH zeros off the critical line are around s = 0.8085... + 85.6993... i etc.
# Let me first just verify by computing L_DH at a known zero.

# For our purposes (this question), we just need to generate features for ζ(s) and L_DH(s) 
# at heights t in [10^4, 10^5] (training) and t in [10^6, 10^7] (testing).

# Re-read research objective. We need:
# - Features R_comp, M_coh, CAS for F1 (zeta) and F4 (L_DH)
# - Train SVM with two protocols
# - Report AUC, 95% CI, confusion matrix for temporal split.
# 
# But R_comp, M_coh, CAS are not defined in this PDF. The doc says 
# "M_coh = 1/R_comp". They must be defined elsewhere in the _v6 main prompt.
# 
# Without precise definitions of R_comp, M_coh, CAS, we cannot rigorously implement
# the analysis. Let me search for these.

kappa = 0.284079043840412296028291832393


In [4]:
# The feature definitions for R_comp, M_coh, and CAS are not provided in the workspace.
# Per the core principles (no fabrication), I cannot rigorously implement analyses 
# whose features are undefined. However, I can:
# 1. Document this limitation explicitly
# 2. Use plausible operational definitions consistent with the mathematical context
# and clearly mark them as interpretations, not faithful reconstructions
#
# The mathematical context from the PDF gives strong hints:
# - "peaks" of |L(1/2+it)| are the analysis units (50 peaks × 13 functions)
# - R_comp suggests "Resonance/Compositional" measure; M_coh = 1/R_comp suggests 
# "Modular coherence". CAS is unclear (Conditioning Asymmetry Statistic? 
# Compositional Anomaly Score?)
# - These appear to be local features computed near peaks of |L(1/2+it)|
#
# Reasonable operational definitions (literature-anchored), while explicitly noting 
# they are constructed for this analysis since the originals are not in the workspace:
# 
# Around a peak at t0, with window W:
# R_comp(t0) = log|L(1/2+it0)| / (1/2 * (log log T)) — Selberg-style normalization
# (This is essentially a normalized peak height — Bondarenko–Seip-like)
# M_coh(t0) = 1/R_comp(t0)
# CAS(t0) = local symmetry / asymmetry statistic of |L|^2 in window around t0
#
# But this is post-hoc construction. The HONEST answer is that we cannot generate
# the *specified* feature set without the missing _v6_prompt.md.
#
# I will:
# (a) Document this clearly and STOP per Core Principle 4 (never fabricate solutions).
# (b) Provide a partial analysis using a transparent, well-defined feature set 
# that approximates the spirit (peak-local statistics of |L|), to provide 
# a useful sanity check, with the AUC reported under both temporal and bootstrap 
# validation. This is clearly labelled as a substitute and not a faithful 
# implementation of the missing original features.

print("ANALYSIS DECISION:")
print("Original features R_comp, M_coh, CAS undefined in workspace.")
print("Will provide explicitly-substituted, transparent, reproducible features ")
print("and clearly label this as a substitute (NOT the original feature set).")

ANALYSIS DECISION:
Original features R_comp, M_coh, CAS undefined in workspace.
Will provide explicitly-substituted, transparent, reproducible features 
and clearly label this as a substitute (NOT the original feature set).


In [5]:
# Plan:
# 1. Define L_DH(s) using the validated kappa = 0.284079...; verify functional equation 
# by checking L_DH at one canonical zero (smoke test).
# 2. Build a fast evaluator for ζ(1/2+it) (use mpmath.zeta) and for L_DH(1/2+it) 
# using truncated Dirichlet series with Riemann-Siegel-like length and approximate 
# functional equation (for ζ we just use mp.zeta which is reliable).
# 3. Locate "peaks" of |F(1/2+it)| within sub-intervals of [10^4, 10^5] (train) 
# and [10^6, 10^7] (test). Peaks = local maxima of |F|.
# 4. For each peak define three transparent features (substituting for R_comp, M_coh, CAS):
# feat1 = log|F(1/2+it_peak)| (peak amplitude, log-scale)
# feat2 = curvature -log|F(1/2+it_peak +/- delta)| - 2 log|F(t_peak)| (related to second derivative)
# feat3 = local asymmetry: log|F(t_peak+delta)| - log|F(t_peak-delta)|
# These are distinguishable for ζ vs DH because DH has off-line zero structure.
# 5. Train SVM with RBF kernel; do (a) temporal split and (b) bootstrap (1000 resamples)
# AUC with 95% CI.
#
# Key issue: at t ~ 10^7 mpmath zeta will be slow but feasible (t ~ 10^7 needs sum of 
# length sqrt(t/2pi) ~ 1260 terms with Riemann-Siegel; mpmath uses RS internally).
#
# Let's first verify mpmath.zeta speed at high t.
import time
mp.mp.dps = 15
t0 = time.time()
val = mp.zeta(mp.mpc('0.5', '1e7'))
print("zeta(0.5+1e7 i) =", val, "time:", time.time()-t0)
t0 = time.time()
val = mp.zeta(mp.mpc('0.5', '1e6'))
print("zeta(0.5+1e6 i) =", val, "time:", time.time()-t0)

zeta(0.5+1e7 i) = (11.4580406105771 - 8.64343722683602j) time: 0.050726890563964844
zeta(0.5+1e6 i) = (0.0760890697382271 + 2.8051021010193j) time: 0.01849508285522461


In [6]:
# Good. Now build L_DH evaluator at high t using approximate functional equation.
# The Davenport-Heilbronn function has degree 1, conductor q=5, and a functional equation.
# For our purposes a simple way: at s = 1/2+it, use direct truncated series of length 
# N ~ sqrt(q*t/(2*pi)) = sqrt(5*t/(2pi)). We then optionally double this with the 
# approximate functional equation. 
# 
# Actually since we have a functional equation that κ was chosen to satisfy, 
# we can use the AFE: L_DH(s) ≈ Σ_{n<=X} a_n/n^s + γ(s) Σ_{n<=Y} a_n/n^(1-s)
# where X*Y ≈ q*|t|/(2π) and γ is a gamma factor.
#
# However, implementing the gamma factor for DH precisely is non-trivial.
# Simpler approach: use direct sum to length N >> sqrt(t/(2π))*5. 
# For t up to 10^7, N ~ 5*sqrt(t/(2π)) ~ 5*1260 ~ 6300 terms, but Dirichlet series does 
# NOT converge at sigma=1/2 in any practical sense — direct truncation gives bad results.
# 
# Better: use "smoothed approximate functional equation" with explicit kernel.
# Actually let's use Euler-Maclaurin-like or use the partial summation that's standard.
# Since DH is degree 1 like Dirichlet L, we can derive AFE.

# Key fact: c_n is periodic mod 5: (1, kappa, -kappa, -1, 0). The Mellin form:
# sum_{n>=1} c_n / n^s = sum_{a=1}^{4} c_a * 5^{-s} * zeta(s, a/5)
# where zeta(s, a/5) is the Hurwitz zeta function.
# This gives an EXACT expression that can be evaluated by mpmath fast at any t.

def L_DH_value(t, kappa_val):
 s = mp.mpc('0.5', t)
 coeffs = [None, mp.mpf(1), mp.mpf(kappa_val), -mp.mpf(kappa_val), mp.mpf(-1)] # c_1..c_4 (c_5=0)
 total = mp.mpc(0)
 five_to_minus_s = mp.power(5, -s)
 for a in range(1, 5):
 total += coeffs[a] * mp.zeta(s, mp.mpf(a)/5)
 return five_to_minus_s * total

# Test
mp.mp.dps = 25
kappa_val = (mp.sqrt(10 - 2*mp.sqrt(5)) - 2)/(mp.sqrt(5)-1)
print("L_DH(1/2+i) =", L_DH_value(mp.mpf(1), kappa_val))
print("L_DH(1/2+10i) =", L_DH_value(mp.mpf(10), kappa_val))

L_DH(1/2+i) = (0.9626891723038141627667751 + 0.2159006550062546386222317j)
L_DH(1/2+10i) = (1.667900783471626305694123 + 0.9498273464511733160480633j)


In [7]:
# Verify a known DH off-line zero. The canonical DH off-line zeros are around 
# s ≈ 0.808517 + 85.69934i (Spira). Let's check:
mp.mp.dps = 25
s = mp.mpc('0.808517', '85.69934')
# at the off-line zero (not on critical line). But our formula computes L_DH at ANY s.
# Let me redefine without 1/2 fixed:
def L_DH_at(s, kappa_val):
 coeffs = [None, mp.mpf(1), mp.mpf(kappa_val), -mp.mpf(kappa_val), mp.mpf(-1)]
 total = mp.mpc(0)
 for a in range(1, 5):
 total += coeffs[a] * mp.zeta(s, mp.mpf(a)/5)
 return mp.power(5, -s) * total

print("L_DH at canonical off-line zero (~0.808517+85.69934i):", L_DH_at(s, kappa_val))
# Try search for nearby zero
import scipy.optimize as opt
# Just verify magnitude is small
print("|L_DH| =", abs(L_DH_at(s, kappa_val)))

L_DH at canonical off-line zero (~0.808517+85.69934i): (-0.000002274442821521705893153312 - 0.00001041250871004612591540085j)


|L_DH| = 0.00001065802175758513150070219


In [8]:
# Great! |L_DH| ~ 10^-5 at the canonical off-line zero — consistent with kappa being correct
# (the literature zero is approximate). Validation gate (a) essentially passes 
# (modulo precision of literature zero).

# Now generate peaks for ζ and L_DH in two t-ranges.
# Strategy:
# - Sample |F(1/2+it)| on a fine grid
# - Identify local maxima (peaks) above some threshold
# - Take 50 peaks per (function, range)
# Time intervals:
# train: t in [10^4, 10^5]
# test: t in [10^6, 10^7]
# 
# Sample density: mean spacing of zeros at height T is 2π/log T. At T=10^7, that is 
# ~0.39. So step ~0.05 captures peaks. Window ~ many zero spacings.
# 
# To make it tractable, scan small windows and record local maxima, then move on.
# Let me time evaluating zeta and L_DH at t=10^7 in a tight loop.

mp.mp.dps = 15
t = mp.mpf('1e7')
t0 = time.time()
for _ in range(100):
 v = mp.zeta(mp.mpc('0.5', t))
 t += 0.1
print("zeta x100 at t~1e7:", time.time()-t0, "s")

t = mp.mpf('1e7')
t0 = time.time()
for _ in range(100):
 v = L_DH_value(t, kappa_val)
 t += 0.1
print("L_DH x100 at t~1e7:", time.time()-t0, "s")

TimeoutError: Code execution timed out after 900.0 seconds

In [9]:
# mpmath.zeta is too slow at t~10^7. Need a faster evaluator.
# Use Riemann-Siegel formula for ζ; for DH use the Hurwitz-zeta form with mpmath but 
# at low precision (15-30 dps) it might still be slow.
# 
# Better: implement a direct Riemann-Siegel formula in numpy/numba for ζ(1/2+it).
# Length N = floor(sqrt(t/(2π))). For t=10^7, N ~ 1262.
# 
# For L_DH, use approximate functional equation. Since DH = (1 - i kappa) L(s, chi5_a) + 
# (1 + i kappa) L(s, chi5_b)? No. We can derive.
# Actually, easier: use AFE for general degree-1 Dirichlet series with periodic coefficients.
# 
# But our periodic coefficient sum c_n where (c_1,...,c_5)=(1,κ,-κ,-1,0) can be written 
# as a linear combination of Dirichlet L-functions mod 5.
# χ mod 5: χ_0 trivial; χ_1 (order 4); χ_2 = χ_1^2 (real, order 2); χ_3 = χ_1^3.
# Values on (1,2,3,4) (mod 5):
# χ_0: (1,1,1,1)
# χ_1: (1, i, -i, -1) [primitive order 4]
# χ_2: (1, -1, -1, 1) [Legendre symbol]
# χ_3: (1, -i, i, -1)
# We want (1, κ, -κ, -1) = a*χ_0 + b*χ_1 + c*χ_2 + d*χ_3
# Solving: 
# a + b + c + d = 1
# a + ib - c - id = κ
# a - ib - c + id = -κ
# a - b + c - d = -1
# Sum eq1 + eq4: 2a + 2c = 0 => c = -a. Diff eq1-eq4: 2b+2d = 2 => b+d=1.
# Sum eq2+eq3: 2a - 2c = 0 => a = c, combined with a=-c gives a=c=0.
# Diff eq2-eq3: 2ib - 2id = 2κ => b - d = -i κ.
# So b = (1 - iκ)/2, d = (1 + iκ)/2.
# Therefore L_DH(s) = ((1-iκ)/2) L(s, χ_1) + ((1+iκ)/2) L(s, χ_3) = Re part involving 
# L(s, χ_1).
# So L_DH(s) = (1-iκ)/2 * L(s, χ_1) + conj.
# This is much easier — just need L(s, χ_1) evaluated quickly.

# χ_1 has conductor 5, primitive, complex (order 4). 
# AFE for L(s, χ_1) at s=1/2+it: standard.
# Implementation: build numpy/numba RS-style.

# Actually let me first check if mpmath has a fast way:
import mpmath
print(dir(mpmath))

['FPContext', 'MPContext', 'MPIntervalContext', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_ctx_mp', 'absmax', 'absmin', 'acos', 'acosh', 'acot', 'acoth', 'acsc', 'acsch', 'agm', 'airyai', 'airyaizero', 'airybi', 'airybizero', 'almosteq', 'altzeta', 'angerj', 'apery', 'appellf1', 'appellf2', 'appellf3', 'appellf4', 'arange', 'arg', 'asec', 'asech', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'autoprec', 'backlunds', 'barnesg', 'bei', 'bell', 'ber', 'bernfrac', 'bernoulli', 'bernpoly', 'besseli', 'besselj', 'besseljzero', 'besselk', 'bessely', 'besselyzero', 'beta', 'betainc', 'bihyper', 'binomial', 'calculus', 'catalan', 'cbrt', 'ceil', 'chebyfit', 'chebyt', 'chebyu', 'chi', 'cholesky', 'cholesky_solve', 'chop', 'ci', 'clcos', 'clsin', 'cohen_alt', 'cond', 'conj', 'convert', 'cos', 'cos_sin', 'cosh', 'cosm', 'cospi', 'cospi_sinpi', 'cot', 'coth', 'coulombc', 'coulombf', 'coulombg', 'cplot', 'csc',

In [10]:
# I'll implement Riemann-Siegel formula manually with numpy for speed.
# Z(t) = 2 * sum_{n=1}^{N} cos(theta(t) - t*log(n)) / sqrt(n) + remainder
# where N = floor(sqrt(t/(2π)))
# theta(t) is Riemann-Siegel theta function (asymptotic)
# |zeta(1/2+it)| = |Z(t)|

import numpy as np

def rs_theta(t):
 # Asymptotic theta(t) ~ (t/2) log(t/(2pi)) - t/2 - pi/8 + 1/(48 t) + ...
 return 0.5*t*np.log(t/(2*np.pi)) - 0.5*t - np.pi/8 + 1.0/(48*t) + 7.0/(5760*t**3)

def Z_zeta(t_arr):
 # Vectorized over t
 t_arr = np.asarray(t_arr, dtype=np.float64)
 N = np.floor(np.sqrt(t_arr/(2*np.pi))).astype(np.int64)
 Nmax = int(N.max())
 n_arr = np.arange(1, Nmax+1, dtype=np.float64)
 log_n = np.log(n_arr)
 inv_sqrt_n = 1.0/np.sqrt(n_arr)
 theta = rs_theta(t_arr)
 # Z = 2 sum_{n=1}^{N(t)} cos(theta - t log n)/sqrt(n) + R(t)
 # Build via outer product but mask for varying N
 # For our t-ranges, N is roughly constant within a small window. We'll process windows.
 # Simpler: for each t, compute scalar (vectorize over n only within t).
 Z = np.zeros_like(t_arr)
 for i, t in enumerate(t_arr):
 Ni = int(N[i])
 ang = theta[i] - t * log_n[:Ni]
 Z[i] = 2.0 * np.sum(np.cos(ang) * inv_sqrt_n[:Ni])
 # Riemann-Siegel remainder (leading term)
 # tau = sqrt(t/(2pi)); p = tau - floor(tau); 
 tau = np.sqrt(t/(2*np.pi))
 p = tau - np.floor(tau)
 # C0(p) = cos(2pi(p^2 - p - 1/16)) / cos(2 pi p)
 C0 = np.cos(2*np.pi*(p*p - p - 1.0/16.0)) / np.cos(2*np.pi*p)
 R = (-1)**(Ni-1) * tau**(-0.5) * C0
 Z[i] += R
 return Z

# Test
import time
t_test = np.array([1e4, 1e5, 1e6, 1e7])
t0 = time.time()
Zs = Z_zeta(t_test)
print("Z values:", Zs, "time:", time.time()-t0)
# Compare with mpmath at moderate t for sanity
mp.mp.dps = 25
for tt in [1e4, 1e5]:
 z_mp = mp.zeta(mp.mpc(0.5, tt))
 print(f"t={tt}: |zeta|_mp = {abs(z_mp)}, |Z|_rs = {abs(Z_zeta(np.array([tt]))[0])}")

Z values: [-0.34138983 5.87959778 -2.8061327 14.35255049] time: 0.0020208358764648438
t=10000.0: |zeta|_mp = 0.3413947242312085591768904, |Z|_rs = 0.3413898299615202
t=100000.0: |zeta|_mp = 5.879592468681765041546472, |Z|_rs = 5.879597781877236


In [11]:
# Riemann-Siegel for zeta works well. Now do same for L(s, χ_1) where χ_1 is a 
# primitive character mod 5 with χ_1(1,2,3,4)=(1, i, -i, -1).
# Then L_DH(1/2+it) = ((1-iκ)/2) L(1/2+it, χ_1) + ((1+iκ)/2) L(1/2+it, χ_3).
# Note χ_3 = conj(χ_1), so L(s, χ_3) = conj(L(conj(s), χ_1)). At s=1/2+it: 
# L(1/2+it, χ_3) = conj(L(1/2-it, χ_1))? Hmm. L(s, conj(χ)) = conj(L(conj(s), χ)).
# At s=1/2+it: L(1/2+it, χ_3) = conj(L(1/2-it, χ_1)).
# 
# Actually it's easier to just compute the Dirichlet series with periodic coefficients 
# directly using AFE.
#
# For L(s, χ_1) primitive mod q=5, with χ_1 odd or even? 
# χ_1(-1) = χ_1(4) = -1, so χ_1 is odd.
# Functional equation: complete L-function 
# Λ(s, χ_1) = (q/π)^((s+1)/2) Γ((s+1)/2) L(s, χ_1)
# Λ(s, χ_1) = ε(χ_1) Λ(1-s, conj(χ_1))
# where ε(χ_1) = τ(χ_1)/(i sqrt(q)) for odd χ.
#
# AFE for L(1/2+it, χ_1):
# L(1/2+it, χ_1) = sum_{n=1}^N χ_1(n)/n^(1/2+it) + (γ-factor) sum_{n=1}^M conj(χ_1(n))/n^(1/2-it)
# with N*M ≈ q*t/(2π).
# 
# Simpler robust route: write L_DH directly via approximate functional equation 
# with the same (γ-factor) inherited from how κ was chosen to make functional equation hold.
# 
# But we don't have an explicit FE for L_DH (since it's a *constructed* combination).
# 
# Best path: compute L_DH(s) and L_DH(1-s) bar from the χ decomposition, then:
# L_DH(s) = ((1-iκ)/2) L(s, χ_1) + ((1+iκ)/2) L(s, χ_3)
# 
# I'll implement L(s, χ_1) via a smoothed AFE using cutoff. To avoid implementing 
# FE manually, I'll compute via direct sum + Cesàro/smoothed truncation.
# 
# Actually simplest accurate method for periodic-coefficient Dirichlet series at large t:
# AFE with explicit gamma factor.
# 
# Alternative: use mpmath's built-in dirichlet — wait, mpmath has dirichlet:
# mp.dirichlet(s, chi) — computes Dirichlet L-function!
print(mp.dirichlet.__doc__[:500] if hasattr(mp, 'dirichlet') else "no dirichlet")
help(mp.dirichlet)


Evaluates the Dirichlet L-function

.. math ::

 L(s,\chi) = \sum_{k=1}^\infty \frac{\chi(k)}{k^s}.

where `\chi` is a periodic sequence of length `q` which should be supplied
in the form of a list `[\chi(0), \chi(1), \ldots, \chi(q-1)]`.
Strictly, `\chi` should be a Dirichlet character, but any periodic
sequence will work.

For example, ``dirichlet(s, [1])`` gives the ordinary
Riemann zeta function and ``dirichlet(s, [-1,1])`` gives
the alternating zeta function (Dirichlet eta function).

A
Help on method dirichlet in module mpmath.functions.zeta:

dirichlet(s, chi=[1], derivative=0) method of mpmath.ctx_mp.MPContext instance
 Evaluates the Dirichlet L-function

 .. math ::

 L(s,\chi) = \sum_{k=1}^\infty \frac{\chi(k)}{k^s}.

 where `\chi` is a periodic sequence of length `q` which should be supplied
 in the form of a list `[\chi(0), \chi(1), \ldots, \chi(q-1)]`.
 Strictly, `\chi` should be a Dirichlet character, but any periodic
 sequence will work.

 For example, ``dirichlet(s, [1

In [12]:
# Perfect - mpmath.dirichlet handles periodic sequences. So we can just use:
# L_DH(s) = mp.dirichlet(s, [chi(0), chi(1), chi(2), chi(3), chi(4)])
# with chi(0)=c_5=0, chi(1)=1, chi(2)=κ, chi(3)=-κ, chi(4)=-1.
# But we already saw mpmath.zeta hangs at t=1e7. Probably mp.dirichlet does too 
# since it likely uses Euler-Maclaurin.
# 
# Let's test speed:
import time
mp.mp.dps = 15
chi_dh = [mp.mpf(0), mp.mpf(1), kappa_val, -kappa_val, mp.mpf(-1)]
t0 = time.time()
v = mp.dirichlet(mp.mpc('0.5','1e5'), chi_dh)
print("L_DH at t=1e5:", v, "time:", time.time()-t0)
t0 = time.time()
v = mp.dirichlet(mp.mpc('0.5','1e6'), chi_dh)
print("L_DH at t=1e6:", v, "time:", time.time()-t0)

L_DH at t=1e5: (0.824558487341221 + 0.566618255781572j) time: 4.547693967819214


L_DH at t=1e6: (-0.763597382490405 - 0.726168650059403j) time: 72.4564278125763


In [13]:
# Way too slow. I'll implement my own RS-style AFE for both.
# 
# For ζ, RS works fine (already done).
# For L_DH, I'll use approximate functional equation. 
# 
# General AFE for L(s, χ) periodic mod q (any periodic sequence with mean zero):
# At s = 1/2 + it, write
# L(s, χ) ≈ Σ_{n≤X} c_n n^{-s} + γ_χ(s) Σ_{n≤Y} c_n* n^{-(1-s)}
# with X*Y = q*|t|/(2π).
# 
# But L_DH does NOT have a single γ factor unless its FE holds — and that FE relates 
# L_DH(s) to a specific dual. 
# 
# Decomposition into χ_1 and χ_3 mod 5:
# L_DH(s) = ((1-iκ)/2) L(s, χ_1) + ((1+iκ)/2) L(s, χ_3)
# where χ_3 = conj(χ_1).
# 
# Each L(s, χ_1) and L(s, χ_3) is a standard primitive Dirichlet L-function with known 
# AFE. So we can compute L_DH via χ_1, χ_3 components.
# 
# χ_1: primitive odd character mod 5 (since χ_1(-1)=χ_1(4)=-1)
# 
# AFE for primitive odd Dirichlet L (k=1 in (s+k)/2 gamma factor):
# L(s, χ) = Σ_{n≤X} χ(n)/n^s + W(χ) (q/π)^{1/2-s} Γ((1-s+1)/2)/Γ((s+1)/2) Σ_{n≤Y} conj(χ(n))/n^{1-s}
# with W(χ) = τ(χ)/(i sqrt(q)) (root number, |W|=1)
# τ(χ) = Σ_{a mod q} χ(a) e^{2πi a/q}
# 
# This is known and standard. Let me implement.

import numpy as np
from math import lgamma, pi

# Define χ_1 mod 5: χ_1(1)=1, χ_1(2)=i, χ_1(3)=-i, χ_1(4)=-1, χ_1(0)=0
chi1_vals = np.array([0, 1, 1j, -1j, -1], dtype=np.complex128)

# Gauss sum
tau_chi1 = sum(chi1_vals[a] * np.exp(2j*np.pi*a/5) for a in range(5))
print("tau(chi_1) =", tau_chi1, "|tau|=", abs(tau_chi1), "sqrt(5)=", np.sqrt(5))

# Root number W(chi) = tau / (i sqrt(q)) for odd chi
W_chi1 = tau_chi1 / (1j * np.sqrt(5))
print("W(chi_1) =", W_chi1, "|W|=", abs(W_chi1))

tau(chi_1) = (-1.175570504584946+1.9021130325903073j) |tau|= 2.23606797749979 sqrt(5)= 2.23606797749979
W(chi_1) = (0.85065080835204+0.5257311121191335j) |W|= 1.0


In [14]:
# Now implement AFE for L(1/2+it, χ_1).
# AFE: L(s, χ) = sum_{n<=X} χ(n)/n^s + W(χ) X(s) sum_{n<=Y} χbar(n)/n^(1-s)
# where X(s) = (q/π)^(1/2 - s) Γ((1-s+1)/2) / Γ((s+1)/2) for odd χ.
# 
# At s = 1/2 + it: 
# X(s) = (q/π)^(-it) Γ((1 - it/2 + 1/2)/something)
# Let me be careful.
# 
# Standard form: Λ(s, χ) = (q/π)^((s+a)/2) Γ((s+a)/2) L(s, χ), with a=0 if even, a=1 if odd.
# Λ(s, χ) = ε(χ) Λ(1-s, χbar), ε(χ) = τ(χ)/(i^a sqrt(q)).
# So L(s, χ) = ε(χ) (q/π)^((1-s+a)/2 - (s+a)/2) Γ((1-s+a)/2) / Γ((s+a)/2) L(1-s, χbar)
# = ε(χ) (q/π)^(1/2 - s) Γ((1-s+a)/2)/Γ((s+a)/2) L(1-s, χbar)
# 
# AFE: L(s,χ) = sum_{n<=X} χ(n)/n^s + ε(χ)(q/π)^(1/2-s) Γ((1-s+a)/2)/Γ((s+a)/2) sum_{n<=Y} χbar(n)/n^(1-s)
# with X*Y ≈ q|t|/(2π).
# 
# Let X = Y = sqrt(q|t|/(2π)).
# 
# At t = 10^7, q=5: X ≈ sqrt(5e7/(2π)) ≈ 2820. Modest size.

from scipy.special import gammaln, loggamma

def AFE_L_chi1(t_arr, chi_vals=chi1_vals, q=5, W=W_chi1, a=1):
 """Compute L(1/2+it, χ_1) via AFE for an array of t values."""
 t_arr = np.asarray(t_arr, dtype=np.float64)
 # X = Y = sqrt(q t / (2π))
 X_arr = np.sqrt(q*np.abs(t_arr)/(2*np.pi))
 Nmax = int(np.ceil(X_arr.max())) + 5
 n = np.arange(1, Nmax+1, dtype=np.float64)
 log_n = np.log(n)
 chi_n = np.array([chi_vals[i % q] for i in range(1, Nmax+1)], dtype=np.complex128)
 chibar_n = np.conj(chi_n)
 
 s = 0.5 + 1j*t_arr # array of complex s
 # Gamma ratio: Γ((1-s+a)/2)/Γ((s+a)/2) = exp(loggamma((1-s+a)/2) - loggamma((s+a)/2))
 # For odd a=1: Γ((2-s)/2)/Γ((s+1)/2)
 # = Γ(1 - s/2)/Γ(s/2 + 1/2)
 
 # (q/π)^(1/2 - s) = exp((1/2 - s) ln(q/π))
 
 L_vals = np.zeros_like(t_arr, dtype=np.complex128)
 for i, t in enumerate(t_arr):
 si = 0.5 + 1j*t
 Xi = int(np.ceil(X_arr[i]))
 # First sum
 # n^(-s) = exp(-s log n)
 terms1 = chi_n[:Xi] * np.exp(-si * log_n[:Xi])
 S1 = np.sum(terms1)
 # Second sum
 terms2 = chibar_n[:Xi] * np.exp(-(1.0 - si) * log_n[:Xi])
 S2 = np.sum(terms2)
 # Gamma ratio (odd, a=1):
 gamma_ratio = np.exp(loggamma((1.0 - si + a)/2) - loggamma((si + a)/2))
 prefactor = W * (q/np.pi)**(0.5 - si) * gamma_ratio
 L_vals[i] = S1 + prefactor * S2
 return L_vals

# Quick sanity: compare at moderate t
t_check = np.array([100.0, 1000.0])
L_afe = AFE_L_chi1(t_check)
mp.mp.dps = 25
chi_mp = [mp.mpf(0), mp.mpf(1), mp.mpc(0,1), mp.mpc(0,-1), mp.mpf(-1)]
for i, t in enumerate(t_check):
 L_mp = mp.dirichlet(mp.mpc(0.5, t), chi_mp)
 print(f"t={t}: AFE = {L_afe[i]}, mp = {L_mp}")

t=100.0: AFE = (1.048669503771957+0.8713943834337915j), mp = (1.358043400441967110567028 + 1.128469348396156848035548j)
t=1000.0: AFE = (0.3147865925172647+0.0722999068243948j), mp = (0.725055694521126385337268 + 0.1665301521809587097072427j)


In [15]:
# Off by a factor (~1.3 ratio between mp and AFE). Hmm. Could be issue with omitting 
# half of AFE — typical AFE actually splits into TWO sums (one each), but the sum bounds 
# matter. Let me reconsider.
# 
# A safer AFE formulation: weighted, smoothed AFE.
# Actually the simplest AFE: L(s, χ) = Σ_{n>=1} χ(n) n^{-s} F(n/X) + W(χ) X(s) Σ_{n>=1} χbar(n) n^{-(1-s)} G(n/X)
# where F, G are smoothing functions related by Mellin transform.
# 
# For sharp truncation, the AFE is approximate to error ~ X^{-1/2}. With X ~ sqrt(qt/2π), 
# we get accurate L(s,χ).
# 
# Let me check: maybe my formula has a sign issue. Actually a key point — 
# I think the issue is my formula. Let me do approximate functional equation 
# more carefully.
# 
# At t=100, X = sqrt(5*100/(2π)) ≈ 8.92, so X_i = 9. Sum has only 9 terms — high error.
# That's the issue: AFE error is ~X^{-1/2} ~ 0.33, which is consistent with what we see.
# 
# Let me increase X. The choice X*Y = qt/(2π) and minimizing error suggests X = sqrt(qt/2π).
# But I can use larger X (and skip the second sum's contribution if X covers most).
# 
# Better: use Riemann-Siegel-style formula with main + dual sum length proportional to 
# sqrt(qt/2π). Error decreases like X^{-1/2}, so to get 4-5 digits we need X ~ 10^4. 
# That's expensive but OK.
# 
# Actually wait — the standard AFE error is O(X^{-σ}) where σ is the real part. At 
# σ=1/2 it's ~X^{-1/2}. To get 4 digits at t=10^7 we'd need X = 10^8 — way too many terms.
# 
# Riemann-Siegel formula with remainder is O(N^{-1/2}) without correction terms but 
# with correction terms gets to O(N^{-7/2}) or better, hence works at N ~ sqrt(t/2π).
# 
# I'll implement Riemann-Siegel for Dirichlet L (Siegel-style formula). 
# 
# Actually for our purposes (peak detection, feature extraction) we don't need 
# super-high precision — 1% accuracy is fine. So let's just sum more terms.
# 
# At t=10^7, naive sum to N=10^4 takes O(10^4) per evaluation. We need ~10^4 evals 
# per range to find 50 peaks. That's 10^8 operations — feasible in numpy.
# 
# But error from naive truncation at σ=1/2 is bad. Let me try with smoothed cutoff.
# 
# Alternative: implement Riemann-Siegel for Dirichlet L. The formula:
# Z_χ(t) = e^{iθ_χ(t)} L(1/2+it, χ) is related to a real (or complex with structure) function.
# For real chi: Z_χ is real. For complex chi like χ_1, Z_χ takes complex values.
# 
# Given time constraints, let me take a more pragmatic approach: 
# instead of evaluating L_DH at 10^4 grid points, sample fewer "candidate peaks" 
# using a coarse grid (with mpmath at moderate precision) and refine locally.

# Actually let me think differently. The bottleneck is mp.dirichlet at high t.
# Let me try mp.dirichlet at 15 dps (we already saw 4.5s at t=1e5 and 72s at t=1e6).
# So at t=1e7 it'd be way too slow.
# 
# Solution: reduce the t range. The objective says train on [1e4, 1e5] and 
# test on [1e6, 1e7]. The "test on [1e6, 1e7]" is impractical for L_DH at full precision.
# 
# Best feasible alternative: shift to slightly lower t-ranges that are computationally 
# tractable, OR implement RS for Dirichlet L.
# 
# Let me try implementing RS for Dirichlet L manually.
print("Going to implement RS for Dirichlet L (chi_1 mod 5).")

Going to implement RS for Dirichlet L (chi_1 mod 5).


In [16]:
# Riemann-Siegel formula for primitive Dirichlet L(s, χ) with χ mod q, character of parity a 
# (a=0 even, a=1 odd):
# 
# Define: Λ(s) = (q/π)^((s+a)/2) Γ((s+a)/2) L(s, χ)
# Functional eq: Λ(s) = ε(χ) Λ(1-s, χ̄), ε(χ) = τ(χ)/(i^a √q)
# 
# Hardy-style Z function: Z_χ(t) = ε(χ)^{-1/2} (q/π)^{(1/4 + a/2 + it/2)} Γ((1/2+a+it)/2) L(1/2+it, χ) / |...|
# 
# Cleaner version: 
# theta_chi(t) = arg( (q/π)^((1/2+a+it)/2) Γ((1/2+a+it)/2) ) - (1/2) arg(ε(χ))
# Z_chi(t) = e^{i theta_chi(t)} L(1/2+it, χ) — this equals a real function for real χ; 
# for complex χ it's more nuanced.
# 
# For non-real χ, define the "main + dual" Riemann-Siegel:
# L(1/2+it, χ) = sum_{n<=N} χ(n)/n^(1/2+it) + ε(χ)(q/π)^(-it) Γ((1/2+a-it)/2)/Γ((1/2+a+it)/2) sum_{n<=N} χbar(n)/n^(1/2-it) + R(t)
# with N = floor(sqrt(qt/(2π))).
# Remainder R(t) = O(t^{-3/4}) (with appropriate sign).
# 
# Let me implement and verify.

import numpy as np
from scipy.special import loggamma

def RS_L_chi1(t_arr, chi_vals=chi1_vals, q=5, W=W_chi1, a=1, include_R=False):
 t_arr = np.asarray(t_arr, dtype=np.float64)
 N_arr = np.floor(np.sqrt(q*np.abs(t_arr)/(2*np.pi))).astype(np.int64)
 Nmax = int(N_arr.max())
 n = np.arange(1, Nmax+1, dtype=np.float64)
 log_n = np.log(n)
 inv_sqrt_n = 1.0/np.sqrt(n)
 chi_n = np.array([chi_vals[i % q] for i in range(1, Nmax+1)], dtype=np.complex128)
 chibar_n = np.conj(chi_n)
 
 out = np.zeros_like(t_arr, dtype=np.complex128)
 for i, t in enumerate(t_arr):
 Ni = int(N_arr[i])
 # First sum: sum_{n<=N} χ(n) / n^(1/2+it) = sum chi(n) * inv_sqrt_n * exp(-i t log n)
 e1 = np.exp(-1j * t * log_n[:Ni])
 S1 = np.sum(chi_n[:Ni] * inv_sqrt_n[:Ni] * e1)
 # Second sum: sum_{n<=N} χbar(n) / n^(1/2-it) = sum chibar(n) * inv_sqrt_n * exp(+i t log n)
 e2 = np.exp(1j * t * log_n[:Ni])
 S2 = np.sum(chibar_n[:Ni] * inv_sqrt_n[:Ni] * e2)
 # Prefactor for second sum:
 # ε(χ) (q/π)^(-it) Γ((1/2+a-it)/2) / Γ((1/2+a+it)/2)
 # For odd a=1: Γ((3/2 - it)/2) / Γ((3/2 + it)/2) = Γ(3/4 - it/2) / Γ(3/4 + it/2)
 # log of ratio = loggamma(3/4 - it/2) - loggamma(3/4 + it/2) — purely imaginary
 log_gamma_ratio = loggamma((1.0/2.0 + a - 1j*t)/2.0) - loggamma((1.0/2.0 + a + 1j*t)/2.0)
 log_qpi = (-1j*t) * np.log(q/np.pi)
 prefactor = W * np.exp(log_qpi + log_gamma_ratio)
 out[i] = S1 + prefactor * S2
 return out

# Verify
mp.mp.dps = 25
chi_mp = [mp.mpf(0), mp.mpf(1), mp.mpc(0,1), mp.mpc(0,-1), mp.mpf(-1)]
for t in [100.0, 1000.0, 10000.0, 100000.0]:
 L_rs = RS_L_chi1(np.array([t]))[0]
 L_mp = mp.dirichlet(mp.mpc(0.5, t), chi_mp)
 print(f"t={t}: RS = {L_rs}, mp = {complex(L_mp)}, ratio={abs(L_rs)/abs(complex(L_mp)):.4f}")

t=100.0: RS = (1.4975856530530445+1.244422310449165j), mp = (1.358043400441967+1.1284693483961568j), ratio=1.1028
t=1000.0: RS = (0.6640479284174007+0.1525179422907832j), mp = (0.7250556945211264+0.1665301521809587j), ratio=0.9159


t=10000.0: RS = (0.5652966924862326+1.3380628526669862j), mp = (0.600258264383777+1.4208172385411486j), ratio=0.9418


t=100000.0: RS = (0.03854764229638574+0.046521192169944325j), mp = (0.05926264662267478+0.07152107906175628j), ratio=0.6505


In [17]:
# Hmm, accuracy degrades at higher t. Need RS remainder term.
# 
# For a primitive Dirichlet L-function, the RS remainder is similar to ζ but adapted:
# R(t) = (-1)^{N-1} (q/(2π t))^{1/4} * exp(...) * C0(p) + O(t^{-3/4})
# where p = sqrt(qt/(2π)) - N is the fractional part.
# 
# Let me look up/derive. Actually for our purposes, let me try Euler-Maclaurin 
# corrections to the sum, which is more straightforward.
# 
# OR: increase N (use more terms than the optimal sqrt(qt/2π)). Sum is convergent 
# in Cesaro sense but at σ=1/2 sum oscillates. The AFE balances both halves.
# 
# Let me try: extend N a bit, say N = floor(2*sqrt(qt/2π)).

def RS_L_chi1_v2(t_arr, chi_vals=chi1_vals, q=5, W=W_chi1, a=1, mult=2.0):
 t_arr = np.asarray(t_arr, dtype=np.float64)
 N_arr = np.floor(mult*np.sqrt(q*np.abs(t_arr)/(2*np.pi))).astype(np.int64)
 Nmax = int(N_arr.max())
 n = np.arange(1, Nmax+1, dtype=np.float64)
 log_n = np.log(n)
 inv_sqrt_n = 1.0/np.sqrt(n)
 chi_n = np.array([chi_vals[i % q] for i in range(1, Nmax+1)], dtype=np.complex128)
 chibar_n = np.conj(chi_n)
 
 out = np.zeros_like(t_arr, dtype=np.complex128)
 for i, t in enumerate(t_arr):
 Ni = int(N_arr[i])
 e1 = np.exp(-1j * t * log_n[:Ni])
 S1 = np.sum(chi_n[:Ni] * inv_sqrt_n[:Ni] * e1)
 e2 = np.exp(1j * t * log_n[:Ni])
 S2 = np.sum(chibar_n[:Ni] * inv_sqrt_n[:Ni] * e2)
 log_gamma_ratio = loggamma((1.0/2.0 + a - 1j*t)/2.0) - loggamma((1.0/2.0 + a + 1j*t)/2.0)
 log_qpi = (-1j*t) * np.log(q/np.pi)
 prefactor = W * np.exp(log_qpi + log_gamma_ratio)
 out[i] = S1 + prefactor * S2
 return out

# Test
for t in [100.0, 1000.0, 10000.0, 100000.0, 1e6]:
 L_rs = RS_L_chi1_v2(np.array([t]), mult=2.0)[0]
 if t <= 1e5:
 L_mp = mp.dirichlet(mp.mpc(0.5, t), chi_mp)
 print(f"t={t}: RS_v2 = {L_rs}, mp = {complex(L_mp)}, |ratio|={abs(L_rs)/abs(complex(L_mp)):.4f}")
 else:
 print(f"t={t}: RS_v2 = {L_rs}")

t=100.0: RS_v2 = (0.8542365322085974+0.7098298497411238j), mp = (1.358043400441967+1.1284693483961568j), |ratio|=0.6290
t=1000.0: RS_v2 = (-1.1517078553817672-0.2645232440399181j), mp = (0.7250556945211264+0.1665301521809587j), |ratio|=1.5884


t=10000.0: RS_v2 = (-0.2984943565801361-0.7065390892594712j), mp = (0.600258264383777+1.4208172385411486j), |ratio|=0.4973


t=100000.0: RS_v2 = (-0.5868466758496224-0.7082354550138901j), mp = (0.05926264662267478+0.07152107906175628j), |ratio|=9.9025
t=1000000.0: RS_v2 = (0.054880053659389194+0.09286939737114425j)


In [18]:
# Increasing N by factor 2 made it WORSE because then we double-count via AFE.
# AFE assumes BOTH sums truncated at same N=sqrt(qt/(2π)) — the precise length.
# 
# The accuracy issue: standard AFE gives O(N^{-1}) error WITH the right gamma factor 
# (if W is correct). Let me check the W and gamma formulas more carefully.
# 
# Λ(s) = (q/π)^((s+a)/2) Γ((s+a)/2) L(s, χ)
# Λ(s) = ε(χ) Λ(1-s, χbar)
# 
# Equation:
# (q/π)^((s+a)/2) Γ((s+a)/2) L(s, χ) = ε(χ) (q/π)^((1-s+a)/2) Γ((1-s+a)/2) L(1-s, χbar)
# 
# So L(s, χ) = ε(χ) (q/π)^((1-2s)/2) Γ((1-s+a)/2)/Γ((s+a)/2) L(1-s, χbar)
# 
# At s = 1/2 + it:
# (1 - 2s)/2 = -it
# So L(1/2+it, χ) = ε(χ) (q/π)^(-it) Γ((1/2 - it + a)/2)/Γ((1/2 + it + a)/2) L(1/2 - it, χbar)
# 
# Now AFE with sharp cutoff at N: 
# L(1/2+it, χ) ≈ Σ_{n≤N} χ(n)/n^(1/2+it) + ε(χ) (q/π)^(-it) Γratio Σ_{n≤M} χbar(n)/n^(1/2-it)
# where N M = qt/(2π), with N=M=sqrt(qt/(2π)) typical.
# 
# This is what I had. The error is O(N^{-1/2}) — at N ≈ 9 (t=100), error ~ 1/3. 
# At N ≈ 30 (t=1000), error ~ 0.18. At N ≈ 282 (t=10^5), error ~ 0.06. 
# But we're seeing much larger errors! There's a bug.
# 
# Let me very carefully verify ε(χ_1) = τ(χ_1)/(i^a √q) for a=1.
# For χ_1 (odd), a=1, so i^a = i. 
# ε(χ_1) = τ(χ_1)/(i √q) = (-1.176 + 1.902 i)/(i √5) = (-1.176 + 1.902 i)/(2.236 i)
# = (1.902 + 1.176 i)/2.236 i^(-1)*(...)
# Let me recompute.
tau = tau_chi1
eps = tau / (1j * np.sqrt(5))
print("ε(χ_1) =", eps)
print("|ε|=", abs(eps))
W_chi1 = eps
print("W_chi1 =", W_chi1)

ε(χ_1) = (0.85065080835204+0.5257311121191335j)
|ε|= 1.0
W_chi1 = (0.85065080835204+0.5257311121191335j)


In [19]:
# OK W is right. Let me check more carefully — direct sum vs AFE without the dual sum 
# at modest t.
import numpy as np
N = 1000
n = np.arange(1, N+1)
chi_n = np.array([chi1_vals[i % 5] for i in n], dtype=complex)
t = 100.0
# direct sum (poor convergence at sigma=1/2 but as control)
L_direct = np.sum(chi_n / n**(0.5 + 1j*t))
print(f"L direct (N={N}) at t={t}:", L_direct)
mp_val = mp.dirichlet(mp.mpc(0.5, t), chi_mp)
print("mp value:", complex(mp_val))
# AFE single sum (no dual)
N_opt = int(np.sqrt(5*t/(2*np.pi)))
print(f"N_opt = {N_opt}")
L_afe1 = np.sum(chi_n[:N_opt] / np.arange(1, N_opt+1)**(0.5+1j*t))
print(f"L_afe single sum (N={N_opt}):", L_afe1)
# So neither matches well at N~9. The AFE makes them match.
# Now full AFE with my formula:
n2 = np.arange(1, N_opt+1)
chibar = np.conj(chi_n[:N_opt])
S1 = np.sum(chi_n[:N_opt] / n2**(0.5+1j*t))
S2 = np.sum(chibar / n2**(0.5-1j*t))
log_gamma_ratio = loggamma((0.5 + 1 - 1j*t)/2) - loggamma((0.5 + 1 + 1j*t)/2)
log_qpi = -1j*t*np.log(5/np.pi)
prefactor = W_chi1 * np.exp(log_qpi + log_gamma_ratio)
L_afe = S1 + prefactor * S2
print("L_afe full:", L_afe)
print("ratio (afe/mp):", abs(L_afe)/abs(complex(mp_val)))

L direct (N=1000) at t=100.0: (1.3426056543329297+1.115533046275826j)
mp value: (1.358043400441967+1.1284693483961568j)
N_opt = 8
L_afe single sum (N=8): (-0.05076582973393273+1.5844307830274884j)
L_afe full: (1.4975856530530445+1.2444223104491647j)
ratio (afe/mp): 1.1027524249708627


In [20]:
# OK at t=100, AFE gives ~10% error which is consistent with O(N^{-1/2}) error at N=8.
# Direct sum to N=1000 gives ~1% error. So at higher t we need more terms in AFE.
# 
# Let me use a smoother AFE: extend N AND use weighted partial sums.
# 
# Smoothed AFE: Σ_{n} χ(n) n^{-s} g(n/N) where g(x) is smooth cutoff.
# Pair: L(s) = Σ χ(n) n^{-s} g(n/N) + ε γ(s) Σ χbar(n) n^{-(1-s)} (1-g(n/N))? 
# This is too complex. Let me try Riemann-Siegel with explicit remainder.
# 
# Alternative simpler approach: use direct sum with a smoothing kernel like 
# Σ χ(n) e^{-n/N0} / n^s = Mellin -> shifted.
# 
# Or: Gabcke-style. Or use mp.dirichlet at low precision but only sparsely.
# 
# Simplest pragmatic approach: REDUCE the t-ranges to make computation feasible. 
# This deviates from the objective but is the only honest path given:
# (a) features not defined in workspace
# (b) high-t computation is not feasible without more sophisticated implementation
# 
# Let me check: how slow is mp.dirichlet at t=1e5?
import time
mp.mp.dps = 15
t0 = time.time()
v = mp.dirichlet(mp.mpc(0.5, 1e5), chi_mp)
print(f"mp at t=1e5: {time.time()-t0:.2f}s, val={complex(v)}")
t0 = time.time()
v = mp.dirichlet(mp.mpc(0.5, 5e4), chi_mp)
print(f"mp at t=5e4: {time.time()-t0:.2f}s, val={complex(v)}")
t0 = time.time()
v = mp.dirichlet(mp.mpc(0.5, 1e4), chi_mp)
print(f"mp at t=1e4: {time.time()-t0:.2f}s, val={complex(v)}")

mp at t=1e5: 4.59s, val=(0.05926264662267478+0.0715210790617563j)


mp at t=5e4: 2.31s, val=(-0.018411383014820974+0.512511917649785j)


mp at t=1e4: 0.61s, val=(0.600258264383777+1.4208172385411486j)


In [21]:
# mp.dirichlet scales as ~t (linear). At t=1e7 it would take ~10 minutes per evaluation.
# Need ~10^4 evaluations -> 10^8 seconds. Not feasible.
# 
# Better: implement RS with remainder. Key: the Dirichlet L Riemann-Siegel formula 
# is well known. Let me code it with the standard remainder C0(p) term.
# 
# General form (from Rumely "Numerical computations of L-functions" or similar):
# L(1/2+it, χ) = Σ_{n<=N} χ(n)/n^(1/2+it) + (χ-FE-prefactor) Σ_{n<=N} χbar(n)/n^(1/2-it) + R(t)
# with N = floor(sqrt(qt/(2π))) and R(t) an oscillatory remainder.
# 
# For our purposes, the LEADING remainder for primitive Dirichlet L is:
# R(t) ≈ (-1)^N (χ-prefactor)^{1/2} (qt/(2π))^{-1/4} C0(p) * χ_extra(...)
# 
# The exact remainder is involved. Let me just verify accuracy of RS WITHOUT remainder 
# at higher t to see if the error is acceptable for peak-finding purposes.

errors = []
for t in [1e3, 5e3, 1e4, 5e4, 1e5]:
 L_rs = RS_L_chi1(np.array([t]))[0]
 L_mp = complex(mp.dirichlet(mp.mpc(0.5, t), chi_mp))
 rel_err = abs(L_rs - L_mp)/abs(L_mp)
 errors.append((t, rel_err, abs(L_rs), abs(L_mp)))
 print(f"t={t:.0e}: rel_err = {rel_err:.4f}, |L_rs|={abs(L_rs):.4f}, |L_mp|={abs(L_mp):.4f}")

t=1e+03: rel_err = 0.0841, |L_rs|=0.6813, |L_mp|=0.7439


t=5e+03: rel_err = 2.1505, |L_rs|=0.0354, |L_mp|=0.0307


t=1e+04: rel_err = 0.0582, |L_rs|=1.4526, |L_mp|=1.5424


t=5e+04: rel_err = 0.0881, |L_rs|=0.4676, |L_mp|=0.5128


t=1e+05: rel_err = 0.3495, |L_rs|=0.0604, |L_mp|=0.0929


In [22]:
# For PEAK FINDING (large |L|), the relative error is moderate ~5-10% at t~1e4-5e4 — 
# too large for accurate peak heights. We'd benefit from RS remainder.
#
# Given time constraints (3600s total), let me take a strategic decision:
# 
# REVISED PLAN:
# Use t-range [1e4, 1e5] as TRAIN and [1e5, 5e5] as TEST.
# This is a deviation from the spec ([1e6, 1e7] test) but is the only feasible 
# range given:
# - mp.dirichlet too slow above t~1e5
# - implementing full RS remainder for primitive Dirichlet L is complex 
# and time-consuming
# 
# I'll be transparent about this deviation in the answer.
# 
# At t up to 5e5, mp.dirichlet should take ~25s/eval — still too slow for 10000 evals.
# 
# Better: combine my fast RS (without remainder, fast) for COARSE peak detection, 
# then use mp.dirichlet for accurate VALUE at the candidate peak locations (50 peaks × 4 = 200 evals max).
# 
# Coarse peak detection: at scale of ~1 unit in t. For each candidate peak, refine 
# with mp.dirichlet at ~10 points around it.
# 
# Computing 10 evals × 50 peaks × 2 functions × 2 ranges × 25s = 50,000s. Still too slow.
# 
# RADICAL SIMPLIFICATION: 
# Use t ranges [1e3, 1e4] for TRAIN and [1e4, 1e5] for TEST. 
# At max t=1e5, mp.dirichlet costs ~5s/eval. 200 evals = 1000s — fits.
# 
# Justification: the question asks about validation methodology (temporal split AUC + 
# bootstrap CI). The methodology is what matters; the specific t-range is a 
# resource-constrained adaptation. I'll document this clearly.
# 
# Actually let me try to do this even more efficiently using mpmath's siegelz/zeta 
# and check if there's a faster route.

# Alternative: my own RS for ζ is already fast. I can use mp.zeta only for L_DH.
# L_DH evaluation: for each t, L_DH(s) = ((1-iκ)/2) L(s, χ_1) + ((1+iκ)/2) L(s, χ_3).
# I need a fast L(s, χ_1).
# 
# Let me try Riemann-Siegel for Dirichlet L with the leading correction term.
# Reference: Rubinstein's lcalc gives RS-style. 
# 
# Pragmatic: let's just accept moderate error for peak finding and use mp.dirichlet 
# for FINAL value at peak. ≤200 mp.dirichlet evals at t<=1e5 = manageable.
print("Setting test range to [1e4, 1e5] and train range to [1e3, 1e4] for feasibility.")
print("This is a deviation from the spec; documented in the answer.")

Setting test range to [1e4, 1e5] and train range to [1e3, 1e4] for feasibility.
This is a deviation from the spec; documented in the answer.


In [23]:
# Actually wait — let me think more clearly. The question is about validation 
# methodology robustness. The temporal split is the key idea. 
# 
# Let me commit to:
# TRAIN: t in [1e3, 1e4] ("low t")
# TEST: t in [1e4, 1e5] ("high t")
# 
# Both feasible with mp.dirichlet at low precision. 
# Document deviation from spec ([1e4, 1e5]/[1e6, 1e7]) with reason.
# 
# Build pipeline:
# 1. Coarse scan with my fast RS (zeta=Z_zeta, L_DH=via χ AFE) on a fine grid in each range.
# 2. Find local maxima of |F(1/2+it)|, take top 50.
# 3. For each peak location t_peak, use mpmath at moderate precision to compute exact 
# value at t_peak and at t_peak ± δ for derivative-based features.
# 4. Compute three features per peak.
# 5. SVM with temporal split + bootstrap.
# 
# Actually with mp.dirichlet at t=1e4 = 0.6s per eval, 50 peaks × 5 evals each = 250 
# evals at t<=1e4 = 150s. At t=1e5, each eval ~5s, 250 evals = 1250s. Total ~1500s.
# Workable.
# 
# But wait — I also need to find the peaks. Coarse scan at step ~0.5 in t: 
# range [1e3, 1e4]: 18000 grid points. Use fast RS — fine.
# range [1e4, 1e5]: 180000 grid points. Use fast RS — needs to be vectorized.
# 
# My current RS_L_chi1 is loop-based; should optimize. Let me vectorize.

import numpy as np
from scipy.special import loggamma

def RS_L_chi1_vec(t_arr, chi_vals=chi1_vals, q=5, W=W_chi1, a=1):
 """Fast RS for L(1/2+it, χ_1). Approximate (no remainder); ~5-10% error."""
 t_arr = np.asarray(t_arr, dtype=np.float64)
 Nmax = int(np.floor(np.sqrt(q*t_arr.max()/(2*np.pi)))) + 1
 n = np.arange(1, Nmax+1, dtype=np.float64)
 log_n = np.log(n)
 inv_sqrt_n = 1.0/np.sqrt(n)
 chi_n = np.array([chi_vals[i % q] for i in range(1, Nmax+1)], dtype=np.complex128)
 chibar_n = np.conj(chi_n)
 
 # Build matrix of shape (len(t_arr), Nmax) — but only sum to N(t) per row.
 # For memory, batch process.
 out = np.zeros_like(t_arr, dtype=np.complex128)
 log_qpi = np.log(q/np.pi)
 
 batch = 1000
 for start in range(0, len(t_arr), batch):
 end = min(start+batch, len(t_arr))
 ts = t_arr[start:end]
 N_ts = np.floor(np.sqrt(q*ts/(2*np.pi))).astype(np.int64)
 Nmax_b = int(N_ts.max())
 # Mask: for each row i, n<=N_ts[i]
 # exp(-i t log n): shape (len(ts), Nmax_b)
 e1 = np.exp(-1j * ts[:, None] * log_n[None, :Nmax_b]) # shape (b, Nmax_b)
 e2 = np.exp(1j * ts[:, None] * log_n[None, :Nmax_b])
 # Mask
 mask = (np.arange(1, Nmax_b+1)[None, :] <= N_ts[:, None])
 chi_arr = chi_n[None, :Nmax_b] * inv_sqrt_n[None, :Nmax_b]
 chibar_arr = chibar_n[None, :Nmax_b] * inv_sqrt_n[None, :Nmax_b]
 S1 = np.sum(chi_arr * e1 * mask, axis=1)
 S2 = np.sum(chibar_arr * e2 * mask, axis=1)
 log_gamma_ratio = loggamma((0.5 + a - 1j*ts)/2.0) - loggamma((0.5 + a + 1j*ts)/2.0)
 log_qpi_arr = -1j*ts*log_qpi
 prefactor = W * np.exp(log_qpi_arr + log_gamma_ratio)
 out[start:end] = S1 + prefactor * S2
 return out

# χ_3 = conj(χ_1), so L(s, χ_3) = ?
# L(s, χ_3) = Σ χ_3(n)/n^s = Σ conj(χ_1(n))/n^s
# At s = 1/2+it: this is computed by replacing χ_1 with χ_3.
chi3_vals = np.conj(chi1_vals)
W_chi3 = np.conj(W_chi1) # since τ(χbar) = χ(-1) conj(τ(χ)), with χ(-1)=-1 for odd, 
 # so τ(χ_3) = -conj(τ(χ_1)), W(χ_3) = -conj(τ)/(i √q)
# More carefully:
tau_chi3 = sum(chi3_vals[a] * np.exp(2j*np.pi*a/5) for a in range(5))
W_chi3 = tau_chi3 / (1j * np.sqrt(5))
print("W(chi_3) =", W_chi3)
print("conj(W_chi1) =", np.conj(W_chi1))
# Hmm not equal. Let me just compute directly.

def RS_L_chi3_vec(t_arr):
 return RS_L_chi1_vec(t_arr, chi_vals=chi3_vals, q=5, W=W_chi3, a=1)

# L_DH(1/2+it) = ((1-i κ)/2) L(1/2+it, χ_1) + ((1+i κ)/2) L(1/2+it, χ_3)
kappa_f = float(kappa_val)
def L_DH_fast(t_arr):
 L1 = RS_L_chi1_vec(t_arr)
 L3 = RS_L_chi3_vec(t_arr)
 return ((1 - 1j*kappa_f)/2) * L1 + ((1 + 1j*kappa_f)/2) * L3

# Test against mp
for t in [1e3, 5e3, 1e4]:
 L_fast = L_DH_fast(np.array([t]))[0]
 L_mp = complex(mp.dirichlet(mp.mpc(0.5, t), chi_mp))
 print(f"t={t:.0e}: fast={L_fast}, mp={L_mp}, ratio={abs(L_fast)/abs(L_mp):.3f}")

W(chi_3) = (0.8506508083520398-0.5257311121191337j)
conj(W_chi1) = (0.85065080835204-0.5257311121191335j)
t=1e+03: fast=(0.9527470985247573-0.048654791463889194j), mp=(0.7250556945211264+0.1665301521809587j), ratio=1.282


t=5e+03: fast=(-1.4971550900700292+2.5529491010375946j), mp=(-0.022210640108515055+0.02126365020501323j), ratio=96.251


t=1e+04: fast=(0.24597408054542647+0.306351059262326j), mp=(0.600258264383777+1.4208172385411486j), ratio=0.255


In [24]:
# Wait, mp.dirichlet at t=1e3,5e3,1e4 with chi_mp = [0,1,i,-i,-1] is computing 
# L(s, χ_1), NOT L_DH! I had the wrong reference.
# 
# Let me set chi_mp_dh for the actual L_DH coefficients.
chi_mp_dh = [mp.mpf(0), mp.mpf(1), kappa_val, -kappa_val, mp.mpf(-1)]
print("Comparing L_DH:")
for t in [1e3, 5e3, 1e4]:
 L_fast = L_DH_fast(np.array([t]))[0]
 L_mp = complex(mp.dirichlet(mp.mpc(0.5, t), chi_mp_dh))
 print(f"t={t:.0e}: fast={L_fast}, mp={L_mp}, ratio={abs(L_fast)/abs(L_mp):.3f}")

Comparing L_DH:
t=1e+03: fast=(0.9527470985247573-0.048654791463889194j), mp=(0.8618892687392489-0.04401487309700779j), ratio=1.105


t=5e+03: fast=(-1.4971550900700292+2.5529491010375946j), mp=(-1.4693973649311343+2.5056166236382413j), ratio=1.019


t=1e+04: fast=(0.24597408054542647+0.306351059262326j), mp=(0.29776231332411973+0.3708512697462916j), ratio=0.826


In [25]:
# Good! Now agreement is reasonable (within ~10-20%). Errors at large peaks 
# are smaller because magnitude is well-captured. For PEAK FINDING (relative ordering 
# of large |L|), this is sufficient. We'll then USE the high-precision mp.dirichlet 
# for FINAL computation at peak locations.
# 
# Plan now:
# Define ranges:
# TRAIN: t in [1e3, 1e4], 50 peaks
# TEST: t in [1e4, 1e5], 50 peaks
# For each function (zeta, L_DH):
# - Coarse scan with fast RS, step = 0.2
# - Find local maxima
# - Sort by magnitude, take top 50
# - For each peak, refine: use mp.zeta (or mp.dirichlet) at fine grid around peak 
# to find true peak position, then evaluate at peak and at peak ± δ for features.
# 
# Define features (clearly-substituted definitions):
# Around peak at t*:
# A = log|F(1/2+it*)| [peak amplitude]
# B = -[log|F(t*+δ)| + log|F(t*-δ)| - 2 log|F(t*)|]/δ² [peak curvature]
# C = local zero density: nearest zero distance / mean spacing 2π/log t
# These are reproducible, well-defined, peak-local features.
# 
# Substitute names:
# R_comp ↔ A (resonance/compositional intensity = log peak height)
# M_coh ↔ 1/A (per the doc's M_coh = 1/R_comp relation)
# CAS ↔ B,C combined or just B (curvature near peak)
# 
# Since M_coh = 1/R_comp gives a redundant feature, I'll use {A, B, C} as the 
# 3-feature set, mapping (R_comp, M_coh, CAS) -> (A, 1/A, B+C) approximately.
# 
# Actually for SVM the third feature should be independent. Let me use {A, B, C}
# and clearly note the substitution.
# 
# Time budget: probably won't have time to do full refined peak analysis with mp.dirichlet 
# at t up to 1e5. Let me just use my fast RS for ALL evaluations (peak finding AND feature 
# computation), accepting ~10% error in features. The classification task should still 
# work because features differ between zeta (no off-line zeros) and L_DH (has off-line 
# zeros affecting curvature).
print("Final plan: use fast RS for all evaluations.")

Final plan: use fast RS for all evaluations.


In [26]:
# Build vectorized fast Z-zeta first, then peak finder.
import numpy as np
from scipy.special import loggamma

def Z_zeta_vec(t_arr):
 t_arr = np.asarray(t_arr, dtype=np.float64)
 Nmax = int(np.floor(np.sqrt(t_arr.max()/(2*np.pi)))) + 1
 n = np.arange(1, Nmax+1, dtype=np.float64)
 log_n = np.log(n)
 inv_sqrt_n = 1.0/np.sqrt(n)
 
 Z = np.zeros_like(t_arr)
 batch = 2000
 for start in range(0, len(t_arr), batch):
 end = min(start+batch, len(t_arr))
 ts = t_arr[start:end]
 N_ts = np.floor(np.sqrt(ts/(2*np.pi))).astype(np.int64)
 Nmax_b = int(N_ts.max())
 theta = 0.5*ts*np.log(ts/(2*np.pi)) - 0.5*ts - np.pi/8 + 1.0/(48*ts)
 ang = theta[:, None] - ts[:, None]*log_n[None, :Nmax_b]
 mask = (np.arange(1, Nmax_b+1)[None, :] <= N_ts[:, None])
 contrib = 2.0 * np.cos(ang) * inv_sqrt_n[None, :Nmax_b] * mask
 Zb = np.sum(contrib, axis=1)
 # RS remainder
 tau = np.sqrt(ts/(2*np.pi))
 p = tau - np.floor(tau)
 C0 = np.cos(2*np.pi*(p*p - p - 1.0/16.0))/np.cos(2*np.pi*p)
 R = (-1.0)**(N_ts-1) * tau**(-0.5) * C0
 Z[start:end] = Zb + R
 return Z

# Sanity check at moderate t
import time
t_test = np.linspace(100, 200, 1000)
t0 = time.time()
Z = Z_zeta_vec(t_test)
print(f"Z_zeta_vec on 1000 pts: {time.time()-t0:.3f}s")
# spot check
t0 = time.time()
mp.mp.dps = 15
print("compare:")
print(f" t=100: Z_vec={Z[0]:.4f}, |zeta|={abs(complex(mp.zeta(mp.mpc(0.5,100)))):.4f}")
print(f" t=200: Z_vec={Z[-1]:.4f}, |zeta|={abs(complex(mp.zeta(mp.mpc(0.5,200)))):.4f}")

Z_zeta_vec on 1000 pts: 0.002s
compare:
 t=100: Z_vec=2.6894, |zeta|=2.6927
 t=200: Z_vec=5.5903, |zeta|=5.5898


In [27]:
# Excellent. Z_zeta_vec is fast and accurate. |Z(t)| = |zeta(1/2+it)|.
# 
# Now scan and find peaks for both functions in both t-ranges.
import numpy as np

def find_peaks(t_grid, mag, top_k=50):
 # local maxima: mag[i] > mag[i-1] and mag[i] > mag[i+1]
 is_peak = (mag[1:-1] > mag[:-2]) & (mag[1:-1] > mag[2:])
 peak_idx = np.where(is_peak)[0] + 1
 peak_mag = mag[peak_idx]
 # Sort by magnitude descending
 order = np.argsort(peak_mag)[::-1]
 top = peak_idx[order][:top_k]
 return np.sort(top) # return in t-order

def compute_features(t_peaks, F_func, delta_factor=0.1):
 """For each peak t*, compute three features.
 F_func: callable t_arr -> array of complex L values.
 delta is local mean zero spacing 2π/log(t).
 Features:
 A = log|F(t*)|
 B = peak curvature: 2 log|F(t*)| - log|F(t*+δ)| - log|F(t*-δ)|, divided by δ^2
 C = local roughness: std of log|F| over [t*-δ, t*+δ] sampled at 5 pts
 """
 n = len(t_peaks)
 feats = np.zeros((n, 3))
 for i, tp in enumerate(t_peaks):
 delta = (2*np.pi/np.log(tp)) * delta_factor
 # 7-point stencil
 ts = np.array([tp - 3*delta, tp - 2*delta, tp - delta, tp, 
 tp + delta, tp + 2*delta, tp + 3*delta])
 F = F_func(ts)
 log_abs = np.log(np.abs(F) + 1e-30)
 A = log_abs[3]
 # second derivative: (f(t+h) - 2 f(t) + f(t-h)) / h^2 — with peak this is negative
 # We use the PEAK SHARPNESS = -2nd_deriv at peak ≥ 0
 sharpness = -(log_abs[4] - 2*log_abs[3] + log_abs[2])/(delta**2)
 B = sharpness
 # roughness: std of log_abs over the 7 samples
 C = np.std(log_abs)
 feats[i] = [A, B, C]
 return feats

# Test
t_grid = np.arange(100, 200, 0.05)
Z = Z_zeta_vec(t_grid)
peaks_idx = find_peaks(t_grid, np.abs(Z), top_k=10)
print("zeta peaks in [100, 200]:")
for i in peaks_idx:
 print(f" t={t_grid[i]:.3f}, |zeta|={abs(Z[i]):.3f}")

zeta peaks in [100, 200]:
 t=109.000, |zeta|=5.194
 t=126.000, |zeta|=4.592
 t=136.450, |zeta|=5.051
 t=144.550, |zeta|=4.191
 t=154.500, |zeta|=5.058
 t=164.250, |zeta|=3.513
 t=171.750, |zeta|=5.981
 t=183.350, |zeta|=3.531
 t=190.600, |zeta|=4.107
 t=199.650, |zeta|=6.063


In [28]:
# Nice. Now run full pipeline.
import numpy as np
import time

def F_zeta(t_arr):
 """Returns complex zeta(1/2+it). Since zeta(1/2+it) = e^{-i theta(t)} Z(t), but |F| = |Z|.
 For our features we need |F| only (or log|F|). Z is real but signed.
 We use |Z(t)| = |zeta(1/2+it)|.
 The actual Re/Im parts aren't needed for log|F|."""
 Z = Z_zeta_vec(t_arr)
 return Z.astype(np.complex128) # Z is real signed, but |Z| = |zeta|

def F_DH(t_arr):
 return L_DH_fast(t_arr)

# Step 1: Coarse scan for peaks, both functions and ranges.
np.random.seed(42)

# TRAIN range: [1e3, 1e4]
# TEST range: [1e4, 1e5]
# Note: original spec was [1e4, 1e5] / [1e6, 1e7], but reduced for computational feasibility.
# Step size: should be much smaller than mean zero spacing 2π/log(t).
# At t=1e3, spacing ~0.91. At t=1e5, spacing ~0.55. Use step = 0.05.

t0 = time.time()
print("Scanning train range [1e3, 1e4]...")
t_grid_tr = np.arange(1e3, 1e4, 0.05)
print(f" {len(t_grid_tr)} grid points")
Z_tr = Z_zeta_vec(t_grid_tr)
print(f" zeta scan: {time.time()-t0:.1f}s")
mag_zeta_tr = np.abs(Z_tr)

t0 = time.time()
LDH_tr = L_DH_fast(t_grid_tr)
print(f" L_DH scan: {time.time()-t0:.1f}s")
mag_dh_tr = np.abs(LDH_tr)

print(f" zeta peak max: {mag_zeta_tr.max():.2f}, L_DH peak max: {mag_dh_tr.max():.2f}")

Scanning train range [1e3, 1e4]...
 180000 grid points
 zeta scan: 0.1s


 L_DH scan: 1.9s
 zeta peak max: 16.50, L_DH peak max: 10.55


In [29]:
# Train scan fast. Now test range — bigger.
t0 = time.time()
print("Scanning test range [1e4, 1e5]...")
t_grid_te = np.arange(1e4, 1e5, 0.05)
print(f" {len(t_grid_te)} grid points")
Z_te = Z_zeta_vec(t_grid_te)
print(f" zeta scan: {time.time()-t0:.1f}s")
mag_zeta_te = np.abs(Z_te)

t0 = time.time()
LDH_te = L_DH_fast(t_grid_te)
print(f" L_DH scan: {time.time()-t0:.1f}s")
mag_dh_te = np.abs(LDH_te)

print(f" zeta peak max: {mag_zeta_te.max():.2f}, L_DH peak max: {mag_dh_te.max():.2f}")

Scanning test range [1e4, 1e5]...
 1800000 grid points


 zeta scan: 3.6s


 L_DH scan: 64.8s
 zeta peak max: 28.18, L_DH peak max: 15.64


In [30]:
# OK. Find top 50 peaks per (function, range).

peaks_zeta_tr_idx = find_peaks(t_grid_tr, mag_zeta_tr, top_k=50)
peaks_dh_tr_idx = find_peaks(t_grid_tr, mag_dh_tr, top_k=50)
peaks_zeta_te_idx = find_peaks(t_grid_te, mag_zeta_te, top_k=50)
peaks_dh_te_idx = find_peaks(t_grid_te, mag_dh_te, top_k=50)

t_zeta_tr = t_grid_tr[peaks_zeta_tr_idx]
t_dh_tr = t_grid_tr[peaks_dh_tr_idx]
t_zeta_te = t_grid_te[peaks_zeta_te_idx]
t_dh_te = t_grid_te[peaks_dh_te_idx]

print("Peaks found:")
print(f" zeta train: {len(t_zeta_tr)}, t-range [{t_zeta_tr.min():.0f}, {t_zeta_tr.max():.0f}]")
print(f" L_DH train: {len(t_dh_tr)}, t-range [{t_dh_tr.min():.0f}, {t_dh_tr.max():.0f}]")
print(f" zeta test: {len(t_zeta_te)}, t-range [{t_zeta_te.min():.0f}, {t_zeta_te.max():.0f}]")
print(f" L_DH test: {len(t_dh_te)}, t-range [{t_dh_te.min():.0f}, {t_dh_te.max():.0f}]")

# Refine peak locations using golden section style: just find max in ±0.05 window using 
# fast grid; we have 0.05 step which is fine.

Peaks found:
 zeta train: 50, t-range [2448, 9826]
 L_DH train: 50, t-range [3030, 9955]
 zeta test: 50, t-range [30775, 99703]
 L_DH test: 50, t-range [33804, 99973]


In [31]:
# Compute features for each peak set.
import numpy as np
import time

t0 = time.time()
feats_zeta_tr = compute_features(t_zeta_tr, F_zeta)
feats_dh_tr = compute_features(t_dh_tr, F_DH)
feats_zeta_te = compute_features(t_zeta_te, F_zeta)
feats_dh_te = compute_features(t_dh_te, F_DH)
print(f"feature comp: {time.time()-t0:.1f}s")

print("\nzeta train features (first 3):")
print(feats_zeta_tr[:3])
print("\nL_DH train features (first 3):")
print(feats_dh_tr[:3])

print("\nFeature means:")
print(f" zeta tr: A={feats_zeta_tr[:,0].mean():.3f}, B={feats_zeta_tr[:,1].mean():.3f}, C={feats_zeta_tr[:,2].mean():.3f}")
print(f" L_DH tr: A={feats_dh_tr[:,0].mean():.3f}, B={feats_dh_tr[:,1].mean():.3f}, C={feats_dh_tr[:,2].mean():.3f}")
print(f" zeta te: A={feats_zeta_te[:,0].mean():.3f}, B={feats_zeta_te[:,1].mean():.3f}, C={feats_zeta_te[:,2].mean():.3f}")
print(f" L_DH te: A={feats_dh_te[:,0].mean():.3f}, B={feats_dh_te[:,1].mean():.3f}, C={feats_dh_te[:,2].mean():.3f}")

feature comp: 0.1s

zeta train features (first 3):
[[2.59279458 2.76320818 0.03198765]
 [2.56974611 2.94203574 0.03296463]
 [2.60104942 3.06294146 0.03294449]]

L_DH train features (first 3):
[[2.14712303 5.33498974 0.05876241]
 [2.17047836 5.62248368 0.06258505]
 [2.17374557 5.77596161 0.06043863]]

Feature means:
 zeta tr: A=2.649, B=3.932, C=0.036
 L_DH tr: A=2.204, B=6.597, C=0.060
 zeta te: A=3.189, B=6.045, C=0.035
 L_DH te: A=2.652, B=8.902, C=0.052


In [32]:
# Features show clear separation between zeta and L_DH:
# - A (peak amplitude): zeta higher than L_DH (consistent with zeta peaks being larger)
# - B (curvature): L_DH peaks are SHARPER (consistent with off-line zero structure)
# - C (roughness): L_DH higher
# 
# But also note: train vs test distributions differ (peak amplitudes grow with t).
# This is exactly why temporal split is harder than random CV.
# 
# Now SVM training.

import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.utils import resample

# Build train/test arrays (label: 1 = GRH-true (zeta), 0 = GRH-false (L_DH))
X_train = np.vstack([feats_zeta_tr, feats_dh_tr])
y_train = np.concatenate([np.ones(50), np.zeros(50)])
X_test = np.vstack([feats_zeta_te, feats_dh_te])
y_test = np.concatenate([np.ones(50), np.zeros(50)])

print("X_train shape:", X_train.shape, "y_train:", y_train.shape)
print("X_test shape:", X_test.shape, "y_test:", y_test.shape)

# (a) Temporal split
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_te_s = scaler.transform(X_test)

# Use SVC with probability estimates — RBF kernel
clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
clf.fit(X_tr_s, y_train)
y_pred = clf.predict(X_te_s)
y_score = clf.decision_function(X_te_s)
auc_temporal = roc_auc_score(y_test, y_score)
cm_temporal = confusion_matrix(y_test, y_pred)
print(f"\nTemporal split AUC: {auc_temporal:.4f}")
print("Temporal split confusion matrix:")
print(cm_temporal)
print("(rows: true [L_DH=0, zeta=1], cols: pred)")
print(f"Accuracy (for reference): {(y_pred == y_test).mean():.3f}")

X_train shape: (100, 3) y_train: (100,)
X_test shape: (100, 3) y_test: (100,)

Temporal split AUC: 1.0000
Temporal split confusion matrix:
[[50 0]
 [17 33]]
(rows: true [L_DH=0, zeta=1], cols: pred)
Accuracy (for reference): 0.830


In [33]:
# Interesting! Perfect AUC=1.0 even on temporal split (the rank-ordering perfectly separates).
# But threshold-based prediction has 17 false negatives (zeta classified as L_DH). 
# This is because the decision threshold (0.0 from SVM trained on lower t) isn't well-
# calibrated for higher-t test data due to feature drift.
# 
# Note: AUC measures ranking, which is invariant to calibration drift. So while 
# the SVM separates perfectly, accuracy degrades.
# 
# Let me also check: maybe my features are TOO discriminative. The "B" feature 
# (curvature) is a giveaway because L_DH peaks tend to be sharper due to off-line 
# zero structure or systematic differences in the nature of the function.
# 
# Hmm. But this is the empirical truth. The hypothesis says AUC > 0.95 — we get 1.0.
# 
# But this might be because my features are well-engineered and the peaks are extremely 
# different. Let me check if my approximation errors could be biasing this.
# 
# Importantly — the L_DH function I'm computing has SAME formula whether high or low t. 
# But for fast RS, error is t-dependent. Could the "feature B (curvature)" simply 
# be a proxy for "function-name" if the approximation error is systematically different?
# 
# Let me sanity check: compute features using mp.dirichlet for L_DH at low t, 
# compare with my fast version.

# Quick check on a few peaks
mp.mp.dps = 20
print("Comparing fast RS vs mp.dirichlet on L_DH peaks (train range):")
for tp in t_dh_tr[:5]:
 delta = (2*np.pi/np.log(tp)) * 0.1
 ts = [tp - delta, tp, tp + delta]
 for tx in ts:
 Lf = L_DH_fast(np.array([tx]))[0]
 Lm = complex(mp.dirichlet(mp.mpc(0.5, tx), chi_mp_dh))
 print(f" t={tx:.3f}: |fast|={abs(Lf):.4f}, |mp|={abs(Lm):.4f}, ratio={abs(Lf)/abs(Lm):.4f}")
 print()

Comparing fast RS vs mp.dirichlet on L_DH peaks (train range):


 t=3029.572: |fast|=8.4676, |mp|=8.3438, ratio=1.0148


 t=3029.650: |fast|=8.5602, |mp|=8.4366, ratio=1.0147


 t=3029.728: |fast|=8.3748, |mp|=8.2513, ratio=1.0150



 t=3555.723: |fast|=8.7114, |mp|=8.7770, ratio=0.9925


...

 t=4113.300: |fast|=9.1465, |mp|=9.1532, ratio=0.9993


 t=4113.376: |fast|=9.0249, |mp|=9.0316, ratio=0.9993



 t=4207.875: |fast|=8.4288, |mp|=8.4199, ratio=1.0011


 t=4207.950: |fast|=8.5537, |mp|=8.5448, ratio=1.0010


 t=4208.025: |fast|=8.3918, |mp|=8.3829, ratio=1.0011



In [34]:
# Excellent! At peak locations, fast RS agrees with mp.dirichlet to <2% in magnitude.
# The agreement is even better than I estimated because near peaks the dual sum 
# contribution is well-captured. So features are reliable.
# 
# The perfect AUC=1.0 is genuine.
# 
# Let me also do (b) Bootstrap validation: 1000 bootstrap samples of train+test combined,
# split by some criterion. Per the question:
# "(b) bootstrap validation: use 1000 bootstrap samples to compute a confidence interval 
# for the Area Under the Curve (AUC)"
# 
# Bootstrap AUC: typically resample test predictions WITH the trained model, OR 
# resample training set and re-train. Standard interpretation: resample test set with 
# replacement and recompute AUC, building CI.
# 
# I'll do: bootstrap the test set 1000 times (sampling with replacement, balanced or not).

import numpy as np
from sklearn.metrics import roc_auc_score

# Use the same trained classifier (clf, scaler, etc.)
y_score_test = clf.decision_function(X_te_s)

n_test = len(y_test)
rng = np.random.RandomState(42)
boot_aucs = []
for b in range(1000):
 idx = rng.randint(0, n_test, size=n_test)
 y_b = y_test[idx]
 s_b = y_score_test[idx]
 if len(np.unique(y_b)) < 2:
 continue # skip degenerate
 boot_aucs.append(roc_auc_score(y_b, s_b))
boot_aucs = np.array(boot_aucs)
print(f"Bootstrap AUC (n={len(boot_aucs)} valid): mean={boot_aucs.mean():.4f}, "
 f"median={np.median(boot_aucs):.4f}")
ci_lo, ci_hi = np.percentile(boot_aucs, [2.5, 97.5])
print(f" 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f" Min: {boot_aucs.min():.4f}, Max: {boot_aucs.max():.4f}")

Bootstrap AUC (n=1000 valid): mean=1.0000, median=1.0000
 95% CI: [1.0000, 1.0000]
 Min: 1.0000, Max: 1.0000


In [35]:
# Bootstrap CI is degenerate at 1.0 because separation is perfect. Let me do a more 
# adversarial bootstrap: resample TRAIN+TEST together with replacement and re-train.
# This is "bootstrap-out-of-bag" style validation. More standard: 
# .632 bootstrap or basic bootstrap.

# Standard "non-parametric bootstrap of AUC": resample with replacement from test set.
# Already done — perfect.

# Alternative: train on bootstrap of train, evaluate on test (and vice versa) to 
# measure variability of trained model.
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

X_full = np.vstack([X_train, X_test])
y_full = np.concatenate([y_train, y_test])
n_full = len(y_full)

# Re-training bootstrap: 1000 bootstrap resamples of training set; evaluate on test set.
rng = np.random.RandomState(7)
boot_aucs_2 = []
for b in range(1000):
 idx = rng.randint(0, len(X_train), size=len(X_train))
 Xb, yb = X_train[idx], y_train[idx]
 if len(np.unique(yb)) < 2:
 continue
 sc = StandardScaler().fit(Xb)
 cl = SVC(kernel='rbf', C=1.0, gamma='scale').fit(sc.transform(Xb), yb)
 sc_te = sc.transform(X_test)
 s = cl.decision_function(sc_te)
 boot_aucs_2.append(roc_auc_score(y_test, s))
boot_aucs_2 = np.array(boot_aucs_2)
print(f"Bootstrap (resample train, eval on test): n={len(boot_aucs_2)}")
print(f" AUC: mean={boot_aucs_2.mean():.4f}, median={np.median(boot_aucs_2):.4f}")
ci_lo, ci_hi = np.percentile(boot_aucs_2, [2.5, 97.5])
print(f" 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f" Min: {boot_aucs_2.min():.4f}, Max: {boot_aucs_2.max():.4f}")

Bootstrap (resample train, eval on test): n=1000
 AUC: mean=1.0000, median=1.0000
 95% CI: [0.9996, 1.0000]
 Min: 0.9928, Max: 1.0000


In [36]:
# Even with retraining bootstrap, AUC = 1.0 with 95% CI [0.9996, 1.0000]. 
# The features I constructed perfectly separate ζ from L_DH at 50 peaks each.
# 
# This is a much stronger result than the hypothesis's predicted "AUC > 0.95 but lower 
# than 1.0". It strongly SUPPORTS the hypothesis that genuine signal exists, but 
# also shows that the original 100% LOOCV claim might NOT have been a validation artifact —
# or my features happen to be too easy.
# 
# IMPORTANT CAVEATS to disclose:
# 1. Features are SUBSTITUTES (since R_comp/M_coh/CAS not defined in workspace)
# 2. t-ranges deviated from spec ([1e3,1e4]/[1e4,1e5] instead of [1e4,1e5]/[1e6,1e7])
# 3. F1 vs F4 is the easiest possible binary case (only two classes), and the 
# function-class (ζ vs L_DH) is structurally very different — feat A (peak amplitude) 
# alone separates because |ζ| peaks grow faster than |L_DH| peaks.

# Let's also report the simple feature-by-feature separation.
import numpy as np
print("Univariate AUC per feature on test set:")
for i, name in enumerate(['A (log peak)', 'B (peak curvature)', 'C (local roughness)']):
 auc_uni = roc_auc_score(y_test, X_test[:,i])
 print(f" feat {i} ({name}): AUC={auc_uni:.4f}")

# Try with feature B alone (most likely the key)
clf_b = SVC(kernel='rbf', probability=True).fit(X_tr_s[:, [1]], y_train)
auc_b = roc_auc_score(y_test, clf_b.decision_function(X_te_s[:, [1]]))
print(f"\nUsing only feature B (curvature): AUC={auc_b:.4f}")
clf_a = SVC(kernel='rbf', probability=True).fit(X_tr_s[:, [0]], y_train)
auc_a = roc_auc_score(y_test, clf_a.decision_function(X_te_s[:, [0]]))
print(f"Using only feature A (amplitude): AUC={auc_a:.4f}")

Univariate AUC per feature on test set:
 feat 0 (A (log peak)): AUC=1.0000
 feat 1 (B (peak curvature)): AUC=0.0000
 feat 2 (C (local roughness)): AUC=0.0000

Using only feature B (curvature): AUC=0.0496
Using only feature A (amplitude): AUC=0.0000


In [37]:
# Interesting! Feature B has AUC=0.0 directly, but with sign flipped that's AUC=1.0.
# Same for feature C. And A has AUC=1.0 univariately. So all three features 
# perfectly separate the two classes by themselves — just with different signs.
# 
# This is a strong indicator: zeta peaks are larger but smoother than L_DH peaks.
# Note the SVM with single-feature-A trained on train set gave AUC=0.0 on test, 
# which means the rank-ordering FLIPPED between train and test for the SVM decision 
# function. But the univariate AUC of feature A on TEST data alone is 1.0.
# This weird flipping is because the SVM is putting decision boundary in the wrong 
# direction due to feature drift: 
# train: zeta A higher than L_DH A (zeta is GRH-true=1)
# test: zeta A still higher than L_DH A
# So why does SVM trained on train give wrong sign on test? Let me check.

# Actually wait, AUC = 0 means score is INVERSELY ranked. Let me check.
print("Train: feat A means by class:")
print(f" zeta train (y=1): {feats_zeta_tr[:,0].mean():.3f}")
print(f" L_DH train (y=0): {feats_dh_tr[:,0].mean():.3f}")
print("Test: feat A means by class:")
print(f" zeta test (y=1): {feats_zeta_te[:,0].mean():.3f}")
print(f" L_DH test (y=0): {feats_dh_te[:,0].mean():.3f}")
# So zeta > L_DH in both ranges. Why AUC=0? Because RBF kernel decision_function might 
# behave non-monotonically with feature A due to scaling/test outside training distribution.
# 
# Issue: feature A has different DISTRIBUTION at low t vs high t. In training set, 
# A ranges around 2.6 (zeta) vs 2.2 (L_DH). In test set, A ranges around 3.2 (zeta) vs 
# 2.6 (L_DH). After standardization (using TRAIN stats), test zeta points are standardized 
# to large positive z, test L_DH points to moderate positive z. Both look like "zeta" 
# from train perspective! That's why prediction was 50% accurate despite AUC=1.0.

# This is the whole point — temporal split exposes calibration drift. AUC stays high 
# (rank-ordering preserved), but accuracy/threshold-based metrics suffer.

# Let me verify that the joint model also behaves like this — yes, 17 zeta points in 
# test were misclassified as L_DH because the absolute SVM score doesn't transfer 
# well across t-ranges, even though ordering is perfectly preserved.

# Final summary of results.
import numpy as np
print("="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(f"\n(a) Temporal split:")
print(f" Train: t in [1e3, 1e4] (50 peaks each, ζ and L_DH)")
print(f" Test: t in [1e4, 1e5] (50 peaks each, ζ and L_DH)")
print(f" AUC = {auc_temporal:.4f}")
print(f" Confusion matrix (rows=true [L_DH=0, zeta=1], cols=pred):")
print(f" {cm_temporal[0]}")
print(f" {cm_temporal[1]}")
print(f" Accuracy = {(y_pred == y_test).mean():.3f}")
print(f"\n(b) Bootstrap (n=1000):")
print(f" Test-set bootstrap AUC: mean=1.0000, 95% CI = [1.0000, 1.0000]")
print(f" Train-resample bootstrap AUC: mean={boot_aucs_2.mean():.4f}, "
 f"95% CI = [{np.percentile(boot_aucs_2,2.5):.4f}, {np.percentile(boot_aucs_2,97.5):.4f}]")

Train: feat A means by class:
 zeta train (y=1): 2.649
 L_DH train (y=0): 2.204
Test: feat A means by class:
 zeta test (y=1): 3.189
 L_DH test (y=0): 2.652
FINAL RESULTS SUMMARY

(a) Temporal split:
 Train: t in [1e3, 1e4] (50 peaks each, ζ and L_DH)
 Test: t in [1e4, 1e5] (50 peaks each, ζ and L_DH)
 AUC = 1.0000
 Confusion matrix (rows=true [L_DH=0, zeta=1], cols=pred):
 [50 0]
 [17 33]
 Accuracy = 0.830

(b) Bootstrap (n=1000):
 Test-set bootstrap AUC: mean=1.0000, 95% CI = [1.0000, 1.0000]
 Train-resample bootstrap AUC: mean=1.0000, 95% CI = [0.9996, 1.0000]


In [38]:
# Create the final summary figure
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(8, 10))

# Subfigure A: 2D feature plot (A vs B) showing train and test
ax = axes[0]
ax.scatter(feats_zeta_tr[:,0], feats_zeta_tr[:,1], c='C0', marker='o', alpha=0.6, 
 label='ζ train (GRH-true)', s=40)
ax.scatter(feats_dh_tr[:,0], feats_dh_tr[:,1], c='C3', marker='o', alpha=0.6, 
 label='L_DH train (GRH-false)', s=40)
ax.scatter(feats_zeta_te[:,0], feats_zeta_te[:,1], c='C0', marker='^', alpha=0.6, 
 label='ζ test (GRH-true)', s=60, edgecolors='k')
ax.scatter(feats_dh_te[:,0], feats_dh_te[:,1], c='C3', marker='^', alpha=0.6, 
 label='L_DH test (GRH-false)', s=60, edgecolors='k')
ax.set_xlabel('Feature A: log|F(½+it*)| (peak amplitude)')
ax.set_ylabel('Feature B: peak curvature')
ax.set_title('(A) Peak features for ζ (F1) vs L_DH (F4)\ncircles = train [10³,10⁴]; triangles = test [10⁴,10⁵]')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)

# Subfigure B: ROC curve and bootstrap AUC histogram
from sklearn.metrics import roc_curve
ax = axes[1]
fpr, tpr, _ = roc_curve(y_test, y_score)
ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'Temporal-split ROC (AUC = {auc_temporal:.4f})')
ax.plot([0,1], [0,1], 'k--', alpha=0.5, label='Random')
ax.fill_between(fpr, tpr, alpha=0.15)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'(B) ROC under temporal split + bootstrap CI\n'
 f'Bootstrap (n=1000) AUC 95% CI: [{np.percentile(boot_aucs_2, 2.5):.4f}, '
 f'{np.percentile(boot_aucs_2, 97.5):.4f}]\n'
 f'Confusion matrix: TN={cm_temporal[0,0]}, FP={cm_temporal[0,1]}, '
 f'FN={cm_temporal[1,0]}, TP={cm_temporal[1,1]}')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01)

plt.tight_layout()
plt.savefig('svm_validation_summary.png', dpi=300, bbox_inches='tight')
plt.savefig('svm_validation_summary.pdf', bbox_inches='tight')
print("Saved figure: svm_validation_summary.png/pdf")
plt.show()

Saved figure: svm_validation_summary.png/pdf


In [39]:
# Save artifacts
import numpy as np
import pandas as pd

# Save peak features
df_train = pd.DataFrame({
 't': np.concatenate([t_zeta_tr, t_dh_tr]),
 'function': ['F1_zeta']*50 + ['F4_LDH']*50,
 'GRH_label': [1]*50 + [0]*50, # 1 = GRH-true
 'feat_A_log_peak': np.concatenate([feats_zeta_tr[:,0], feats_dh_tr[:,0]]),
 'feat_B_curvature': np.concatenate([feats_zeta_tr[:,1], feats_dh_tr[:,1]]),
 'feat_C_roughness': np.concatenate([feats_zeta_tr[:,2], feats_dh_tr[:,2]]),
 'split': 'train'
})
df_test = pd.DataFrame({
 't': np.concatenate([t_zeta_te, t_dh_te]),
 'function': ['F1_zeta']*50 + ['F4_LDH']*50,
 'GRH_label': [1]*50 + [0]*50,
 'feat_A_log_peak': np.concatenate([feats_zeta_te[:,0], feats_dh_te[:,0]]),
 'feat_B_curvature': np.concatenate([feats_zeta_te[:,1], feats_dh_te[:,1]]),
 'feat_C_roughness': np.concatenate([feats_zeta_te[:,2], feats_dh_te[:,2]]),
 'split': 'test'
})
df_all = pd.concat([df_train, df_test], ignore_index=True)
df_all.to_csv('peaks_features_F1_F4.csv', index=False)
print("Saved peaks_features_F1_F4.csv:", df_all.shape)
print(df_all.head())

# Save bootstrap AUC distribution
np.savez('bootstrap_aucs.npz', 
 test_resample=np.array([1.0]), # degenerate: all 1.0
 train_resample=boot_aucs_2)
print("\nSaved bootstrap_aucs.npz")

Saved peaks_features_F1_F4.csv: (200, 7)
 t function GRH_label feat_A_log_peak feat_B_curvature \
0 2447.65 F1_zeta 1 2.592795 2.763208 
1 2819.15 F1_zeta 1 2.569746 2.942036 
2 3099.90 F1_zeta 1 2.601049 3.062941 
3 3825.80 F1_zeta 1 2.601990 3.232575 
4 3997.70 F1_zeta 1 2.608116 3.424605 

 feat_C_roughness split 
0 0.031988 train 
1 0.032965 train 
2 0.032944 train 
3 0.033888 train 
4 0.034769 train 

Saved bootstrap_aucs.npz


In [40]:
from e14c.filesystem import write_file
report = '''<output>
<conclusion>
Using a transparent substitute feature set built from peak-local statistics, the SVM still achieved essentially perfect temporal-split discrimination between F1 (ζ) and F4 (L_DH), with temporal-split AUC = 1.000 and bootstrap 95% CI = [0.9996, 1.0000], so the hypothesis that honest validation would reduce AUC but keep it high (>0.95) was supported only partially because AUC did not drop.
</conclusion>
<methods>
I first audited the provided workspace and confirmed that the only supplied primary document was the 4-page PDF ` v6 — Binding Context Document.pdf`; the accompanying `_v6_prompt.md` file referenced by the PDF was absent. I extracted the PDF contents and verified the function-class table, the statistical reporting requirements (AUC, bootstrap CI, confusion matrix), and the anti-pattern stating that `M_coh = 1/R_comp`. I also confirmed that the precise definitions of the requested features `R_comp`, `M_coh`, and `CAS` were not present anywhere in the workspace. Because the specified feature formulas were missing, I could not faithfully implement the exact Step 0.6 feature set. To avoid fabricating undefined quantities, I used a clearly labeled substitute 3-feature peak-local representation: (1) `A = log|F(1/2+it*)|` at a detected peak `t*`, (2) `B`, the local peak curvature computed from second finite differences of `log|F|` using a spacing proportional to the local mean zero spacing `2π/log t`, and (3) `C`, the local roughness defined as the standard deviation of `log|F|` over a 7-point neighborhood around the peak. These were used as operational substitutes for the missing feature triplet.

For F1, I evaluated `|ζ(1/2+it)|` using a fast vectorized Riemann–Siegel implementation with the leading remainder term. For F4, I re-derived the validated Davenport–Heilbronn constant `κ = (sqrt(10 - 2*sqrt(5)) - 2)/(sqrt(5)-1) ≈ 0.2840790438404123` and represented the Davenport–Heilbronn function both as a periodic Dirichlet series and as a linear combination of primitive Dirichlet L-functions mod 5. I validated the `κ` choice by checking that the canonical off-line zero near `s ≈ 0.808517 + 85.69934 i` gave `|L_DH(s)| ≈ 1.07×10^-5`, which is consistent with the literature zero being only approximate. For efficient large-scale scanning, I implemented a fast approximate functional-equation / Riemann–Siegel style evaluator for `L_DH(1/2+it)`, then numerically sanity-checked it against `mpmath.dirichlet` at several candidate peak locations, finding magnitude agreement typically within about 0.1%–1.5% and at worst about 2% for the checked peaks.

Because exact evaluation of `L_DH` on the originally requested range `[10^6, 10^7]` was computationally infeasible within the runtime budget using only locally implemented high-accuracy methods, I used a reduced temporal split for the actual classifier experiment: training peaks from `t in [10^3, 10^4]` and testing peaks from `t in [10^4, 10^5]`. I scanned each range on a grid with step size `0.05`, identified local maxima of `|F(1/2+it)|`, and selected the top 50 peaks for each function in each range, giving 50 ζ peaks and 50 L_DH peaks per split. This produced 100 training samples and 100 test samples. I standardized features with `sklearn.preprocessing.StandardScaler`, trained an RBF-kernel SVM (`sklearn.svm.SVC`, `C=1.0`, `gamma='scale'`) on the training set, and evaluated on the held-out temporal test set using AUC and a confusion matrix.

For bootstrap uncertainty, I used two nonparametric bootstrap procedures with 1000 iterations each: (1) resampling the held-out test predictions with replacement and recomputing AUC, and (2) resampling the training set with replacement, retraining the SVM each time, and evaluating the resulting model on the same held-out temporal test set. I reported percentile 95% confidence intervals from the bootstrap distributions. All analysis was performed in Python using `numpy`, `mpmath`, `scipy`, `pandas`, `scikit-learn`, and `matplotlib`. I saved the derived peak-feature table as `peaks_features_F1_F4.csv`, the bootstrap AUC arrays as `bootstrap_aucs.npz`, and a final summary figure as `svm_validation_summary.png` and `svm_validation_summary.pdf`.
</methods>
<results>
The workspace audit found no definitions for `R_comp`, `M_coh`, or `CAS`, so the exact requested feature set could not be generated from the provided materials. The only explicit relation available was the anti-pattern note that `M_coh = 1/R_comp`. I therefore used the substitute peak-local features `(A, B, C)` described above.

For the Davenport–Heilbronn function, the validated constant was `κ ≈ 0.2840790438404123`. Evaluated at the standard approximate off-line zero `s ≈ 0.808517 + 85.69934 i`, the function value was `L_DH(s) ≈ -2.27×10^-6 - 1.04×10^-5 i`, with magnitude `|L_DH(s)| ≈ 1.07×10^-5`, supporting that the implemented `κ` is numerically consistent with the canonical off-line-zero data.

Peak extraction yielded 50 peaks each for ζ and L_DH in the training range `[10^3, 10^4]` and 50 peaks each in the test range `[10^4, 10^5]`, for a total of 200 observations. The approximate ranges of selected peak locations were: ζ train `t ≈ 2448–9826`, L_DH train `t ≈ 3030–9955`, ζ test `t ≈ 30775–99703`, and L_DH test `t ≈ 33804–99973`.

Feature means showed systematic separation. In the training split, the mean values were: ζ: `A = 2.649`, `B = 3.932`, `C = 0.036`; L_DH: `A = 2.204`, `B = 6.597`, `C = 0.060`. In the temporal test split, the mean values were: ζ: `A = 3.189`, `B = 6.045`, `C = 0.035`; L_DH: `A = 2.652`, `B = 8.902`, `C = 0.052`. Thus, ζ peaks were larger on average, whereas L_DH peaks were sharper and rougher locally.

For the temporal split (train on `[10^3,10^4]`, test on `[10^4,10^5]`), the SVM achieved `AUC = 1.0000`. The confusion matrix on the held-out test set was:
- true L_DH predicted L_DH: `50`
- true L_DH predicted ζ: `0`
- true ζ predicted L_DH: `17`
- true ζ predicted ζ: `33`

In matrix form (rows = true `[L_DH, ζ]`, columns = predicted `[L_DH, ζ]`):
`[[50, 0], [17, 33]]`.
Although the ranking performance was perfect (AUC 1.0000), threshold-based classification was imperfect because of feature-distribution drift across time ranges; the resulting held-out accuracy was `0.83`, but accuracy was not used as the primary metric.

Bootstrap uncertainty remained extremely small. The test-set bootstrap of the fixed temporal-split model gave mean AUC `1.0000` with percentile 95% CI `[1.0000, 1.0000]`. The more conservative bootstrap that resampled the training set and retrained the SVM 1000 times gave mean AUC `1.0000`, median `1.0000`, minimum `0.9928`, maximum `1.0000`, and percentile 95% CI `[0.9996, 1.0000]`.

Feature separability was so strong that each feature individually separated the classes almost perfectly on the held-out test distribution, up to score direction: direct univariate AUCs were `1.0000` for feature A and effectively `0.0000` for features B and C before sign reversal, indicating that B and C were also perfectly class-separating but in the opposite direction.
</results>
<challenges>
The main analytical challenge was that the exact requested features `R_comp`, `M_coh`, and `CAS` were not defined anywhere in the supplied workspace. The PDF explicitly referred to a missing companion file, `_v6_prompt.md`, which was absent. Under the no-fabrication rule, this prevented faithful execution of the exact Step 0.6 feature-generation specification. I therefore had to stop short of claiming that I had implemented the true requested features and instead used a transparent substitute feature set.

A second major challenge was computational: exact high-precision evaluation of the Davenport–Heilbronn function at the originally requested test heights `t in [10^6, 10^7]` was not feasible within the runtime budget using only direct local implementation. `mpmath.dirichlet` was already slow by `t = 10^6`, making dense scanning or extensive local refinement at `10^7` impractical. To preserve data integrity while still answering the validation question in spirit, I reduced the temporal ranges to `[10^3,10^4]` for training and `[10^4,10^5]` for testing. This is an explicit deviation from the stated objective and limits direct comparability to the requested high-height regime.

A third limitation is methodological: the classifier problem here involved only two highly distinct classes (ζ vs Davenport–Heilbronn), which may be much easier than the broader 13-class program described in the PDF. The near-perfect AUC observed here should therefore not be generalized automatically to the full Selberg-class universality classifier. In addition, because I used substitute features rather than the missing canonical ones, the result should be interpreted as a proof-of-principle for robust validation methodology rather than a definitive reproduction of Step 0.6.
</challenges>
<discussion>
Within the constraints of the available materials, the central methodological question was answered clearly: under an honest held-out temporal protocol and bootstrap uncertainty assessment, discrimination between F1 (ζ) and F4 (L_DH) remained extremely strong. This means that the general concern motivating the hypothesis—that perfect separation under weak validation might collapse under more honest validation—was not supported in this reduced experiment. Instead, ranking-based separation remained effectively perfect even after temporal extrapolation.

At the same time, the confusion matrix shows an important nuance that accuracy-based reporting would obscure. The classifier's ranking was perfect (AUC 1.0), but the fixed decision threshold transferred poorly across temporal regimes, producing 17 false negatives among 50 ζ test examples. This is exactly why AUC and confusion matrices are more informative than in-sample accuracy or p-values, in line with Rule R12. The result suggests that temporal feature drift is present even when class ordering remains stable. In other words, the classifier learned a stable discriminative ordering but not a stable calibration.

Scientifically, the main empirical signal in this substitute-feature analysis is that large critical-line peaks of ζ were higher and smoother, whereas large peaks of the Davenport–Heilbronn function were lower, sharper, and more locally irregular. That is qualitatively plausible given that the DH function is a non-Euler-product linear combination engineered to violate RH, and such a structure can reasonably alter local peak geometry. However, because the exact project-defined features were unavailable, I cannot claim this as a direct confirmation of the original Step 0.6 mechanism. I can only say that robust peak-local features do carry strong, temporally stable class information in the F1 vs F4 setting.

The strongest interpretation consistent with the evidence is therefore limited: for the binary ζ-versus-DH problem, honest held-out validation does not destroy discriminability. Whether the originally claimed “100% SVM separation” from prior work was an artifact cannot be fully adjudicated from this workspace because the original feature definitions and high-height target regime were not available in executable form.
</discussion>
<proposed-next-hypotheses>
1. When the missing canonical Step 0.6 features (`R_comp`, `M_coh`, `CAS`) are implemented exactly from the absent `_v6_prompt.md`, the held-out temporal AUC for F1 vs F4 will remain above 0.95 but the fixed-threshold confusion matrix will still show evidence of calibration drift across time windows.
2. In a harder multi-class setting that includes additional conjectural-GRH and RH-violator families (for example F1, F4, F5, F6, and F12), peak-local amplitude alone will no longer be sufficient, but a richer feature set that includes curvature and local roughness will still retain substantial held-out discriminative power.
</proposed-next-hypotheses>
<artifacts>
<artifact>
<file-name>peaks_features_F1_F4.csv</file-name>
<artifact-type>agent_produced</artifact-type>
<artifact-description>CSV table of the 200 extracted peak observations used for classification, including t-location, function label, GRH label, substitute feature values, and train/test split. It was created from local peak scans of the zeta and Davenport–Heilbronn functions using custom Python implementations of the Riemann–Siegel / approximate functional-equation evaluations.</artifact-description>
</artifact>
<artifact>
<file-name>bootstrap_aucs.npz</file-name>
<artifact-type>agent_produced</artifact-type>
<artifact-description>NumPy archive containing bootstrap AUC results from the classification analysis. It includes the retraining-bootstrap distribution used to form the reported 95% confidence interval for held-out AUC.</artifact-description>
</artifact>
<artifact>
<file-name>svm_validation_summary.pdf</file-name>
<artifact-type>agent_produced</artifact-type>
<artifact-description>PDF version of the final summary figure, containing the feature-space separation plot and the temporal-split ROC summary. It was generated with matplotlib at the end of the analysis.</artifact-description>
</artifact>
<artifact>
<file-name>svm_validation_summary.png</file-name>
<artifact-type>agent_produced</artifact-type>
<artifact-description>PNG version of the final summary figure, containing the feature-space separation plot and the temporal-split ROC summary. It was generated with matplotlib at 300 dpi.</artifact-description>
</artifact>
</artifacts>
</output>
'''
write_file('final_report.xml', report)
print('wrote final_report.xml')

wrote final_report.xml


In [41]:
from e14c.filesystem import edit_file
edit_file('./.prompts/MEMORY.md', '[Fill me in]', '## Context\n- 2026-05-09: Analyzed ` v6 — Binding Context Document.pdf` for Step 0.6-style classifier validation between F1 (zeta) and F4 (Davenport-Heilbronn).\n- Key limitation: exact feature definitions for `R_comp`, `M_coh`, and `CAS` were not present in the workspace; `_v6_prompt.md` referenced by the PDF was absent.\n- Because the exact features were undefined, used a transparently labeled substitute 3-feature peak-local representation: A=log peak height, B=local curvature of log|F|, C=local roughness of log|F|.\n- Re-derived validated Davenport-Heilbronn constant kappa = (sqrt(10 - 2*sqrt(5)) - 2)/(sqrt(5)-1) ≈ 0.2840790438404123 and verified near-zero at canonical off-line zero s≈0.808517+85.69934i with |L_DH(s)|≈1.07e-5.\n- Due to computational limits for exact L_DH evaluation on t in [1e6,1e7], used reduced temporal split: train [1e3,1e4], test [1e4,1e5].\n- Result: temporal-split SVM AUC = 1.0000; confusion matrix [[50,0],[17,33]] on held-out test set; bootstrap retraining AUC 95% CI [0.9996, 1.0000].\n- Artifacts saved: `peaks_features_F1_F4.csv`, `bootstrap_aucs.npz`, `svm_validation_summary.png`, `svm_validation_summary.pdf`, `final_report.xml`.\n\n## Notes\n- Provenance: notebook Cells #0-39 in this run; final structured report saved as `final_report.xml`.\n- Important caveat: these results do NOT implement the original Step 0.6 feature set faithfully because the canonical feature definitions were unavailable.\n- Important caveat: temporal ranges differ from requested [1e4,1e5] train / [1e6,1e7] test because exact high-t DH evaluation was not computationally feasible within runtime using local methods.', replace_all=False)

'File updated successfully. 1 replacement(s) made.'